# Drive → BigQuery

Loads your Google Drive into BigQuery project `pelagic-gist-505800-b9`, embeds it for semantic search, then crosses it against itself so connections surface without being queried for. Five datasets:

| Dataset | Contents |
|---|---|
| `drive_raw.file_manifest` | One row per file in Drive — the census. Every file appears whether it loaded, failed, or was excluded. Nothing disappears silently. |
| `drive_tables.<name>` | One table per *family* of tabular files. |
| `drive_documents` | Extracted text from PDF, Word, PowerPoint, txt, md — plus `document_chunks`, the passages that get embedded. |
| `drive_vectors.embeddings` | One vector table over documents, spreadsheet rows **and** filenames, with a `search()` function. |
| `drive_graph` | `cross_links` (chunk↔chunk edges), `entity_mentions`, `entities` — the corpus crossed against itself. |
| `drive_insights` | Seven views over that: bridges, indirect relations, timelines, gaps, health — plus an `ask()` function. |

**Why the grouping matters.** A Takeout/Fitbit export is thousands of files that are really a few dozen tables sharded by day:

```
heart_rate_2026-03-23.csv
heart_rate_2026-03-24.csv
heart_rate_2026-04-05.csv
```

Loading one table per file gives you 1000+ useless single-day tables. These are grouped into one table, with the date kept as a `_src_date` column. On the first 100 CSVs in this Drive: **100 files → 28 tables.**

**Why Colab.** Your Drive mounts as ordinary files, so there is no API pagination and no download quota, and BigQuery authenticates as *you* — no service account, no JSON key, nothing to paste.

Run the cells in order. Nothing is written to BigQuery until step 5, and step 4 shows the full plan first. **Step 6 gives you real insights with no setup and no cost** — everything after it is optional.

## 1. Install dependencies

In [ ]:
%pip install --quiet google-cloud-bigquery google-api-python-client \
    pandas pyarrow openpyxl xlrd pypdf python-docx python-pptx
print('dependencies installed')

## 2. Write out the pipeline

In [ ]:
# The pipeline modules, embedded so this notebook needs no clone step.
# Generated by build_notebook.py -- edit the .py files, not this cell.
import pathlib, textwrap

MODULES = {}

MODULES['classify.py'] = r'''__CLASSIFY_PY__
"""Filename/MIME classification and table-family grouping.

The central idea: a Google Takeout / Fitbit export contains thousands of files
that are really *one table each, sharded by date*. `heart_rate_2026-04-05.csv`
and `heart_rate_2026-07-05.csv` are not two tables, they are two days of one
table. Grouping them into "families" is what turns 1000+ loose files into a few
dozen queryable tables.
"""

from __future__ import annotations

import posixpath
import re
from dataclasses import dataclass, field

# MIME types we can parse into rows.
TABULAR_MIMES = {
    "text/csv": "csv",
    "text/tab-separated-values": "tsv",
    "application/vnd.ms-excel": "xls",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": "xlsx",
    "application/vnd.google-apps.spreadsheet": "gsheet",
    "application/json": "json",
    "application/x-ndjson": "ndjson",
}

# MIME types we extract text from.
DOCUMENT_MIMES = {
    "application/pdf": "pdf",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": "docx",
    "application/msword": "doc",
    "application/vnd.google-apps.document": "gdoc",
    "application/vnd.openxmlformats-officedocument.presentationml.presentation": "pptx",
    "application/vnd.google-apps.presentation": "gslides",
    "text/plain": "txt",
    "text/markdown": "md",
    "text/html": "html",
    "application/rtf": "rtf",
}

# Extensions win when Drive reports a useless MIME type (very common for
# Takeout uploads, which often arrive as application/octet-stream).
EXTENSION_OVERRIDES = {
    "csv": ("tabular", "csv"),
    "tsv": ("tabular", "tsv"),
    "xlsx": ("tabular", "xlsx"),
    "xlsm": ("tabular", "xlsx"),
    "xls": ("tabular", "xls"),
    "json": ("tabular", "json"),
    "ndjson": ("tabular", "ndjson"),
    "jsonl": ("tabular", "ndjson"),
    "pdf": ("document", "pdf"),
    "docx": ("document", "docx"),
    "doc": ("document", "doc"),
    "pptx": ("document", "pptx"),
    "txt": ("document", "txt"),
    "md": ("document", "md"),
    "html": ("document", "html"),
    "htm": ("document", "html"),
    "rtf": ("document", "rtf"),
}

# A mounted Drive reports no MIME type, so media has to be recognised by
# extension or it all lands in the "other" bucket.
MEDIA_EXTENSIONS = {
    "jpg", "jpeg", "png", "gif", "bmp", "tif", "tiff", "webp", "heic", "heif",
    "svg", "ico", "raw", "cr2", "nef",
    "mp4", "mov", "avi", "mkv", "webm", "wmv", "flv", "m4v", "mpg", "mpeg",
    "mp3", "wav", "aac", "flac", "ogg", "m4a", "wma", "aiff",
    "ttf", "otf", "woff", "woff2", "eot",
}

FOLDER_MIME = "application/vnd.google-apps.folder"

MEDIA_PREFIXES = ("image/", "video/", "audio/", "font/")

# Trailing shard tokens stripped to find the family stem, longest-first so that
# `_2026-05-01` is consumed before the bare `_2026` rule can nibble at it.
_SHARD_PATTERNS = [
    r"[ _-]+\d{4}[-_]\d{2}[-_]\d{2}(?:[ _-]?\d{2}[-_:]\d{2}(?:[-_:]\d{2})?)?$",
    r"[ _-]+\d{4}[-_]\d{2}$",
    r"[ _-]+\d{8}$",
    r"[ _-]+\d{4}$",
    r"\s*\((\d+)\)$",
    r"[ _-]+copy$",
    r"[ _-]+final$",
    r"[ _-]+v\d+$",
    r"[ _-]+part[ _-]?\d+$",
    r"[ _-]+\d+of\d+$",
]

_DATE_IN_NAME = re.compile(r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})")


def classify(name: str, mime_type: str) -> tuple[str, str]:
    """Return ``(kind, fmt)`` where kind is tabular/document/media/other/folder."""
    if mime_type == FOLDER_MIME:
        return "folder", "folder"

    ext = extension_of(name)
    # Drive's MIME is authoritative for native Google types; otherwise the
    # extension is more trustworthy than octet-stream.
    if mime_type.startswith("application/vnd.google-apps."):
        if mime_type in TABULAR_MIMES:
            return "tabular", TABULAR_MIMES[mime_type]
        if mime_type in DOCUMENT_MIMES:
            return "document", DOCUMENT_MIMES[mime_type]
        return "other", mime_type.rsplit(".", 1)[-1]

    if ext in EXTENSION_OVERRIDES:
        return EXTENSION_OVERRIDES[ext]
    if mime_type in TABULAR_MIMES:
        return "tabular", TABULAR_MIMES[mime_type]
    if mime_type in DOCUMENT_MIMES:
        return "document", DOCUMENT_MIMES[mime_type]
    if mime_type.startswith(MEDIA_PREFIXES):
        return "media", mime_type.split("/", 1)[0]
    if ext in MEDIA_EXTENSIONS:
        return "media", ext
    return "other", ext or "unknown"


def extension_of(name: str) -> str:
    base = posixpath.basename(name)
    if "." not in base:
        return ""
    return base.rsplit(".", 1)[-1].lower()


def strip_extension(name: str) -> str:
    base = posixpath.basename(name)
    if "." in base and extension_of(base):
        return base.rsplit(".", 1)[0]
    return base


def family_stem(name: str) -> str:
    """Strip date/version shard suffixes to get the shared stem of a family."""
    stem = strip_extension(name)
    changed = True
    while changed:
        changed = False
        for pattern in _SHARD_PATTERNS:
            new = re.sub(pattern, "", stem, flags=re.IGNORECASE)
            if new != stem and new.strip(" _-"):
                stem = new
                changed = True
    return stem.strip(" _-")


def shard_date(name: str) -> str | None:
    """Best-effort ISO date pulled from a filename, for the ``_src_date`` column."""
    match = _DATE_IN_NAME.search(strip_extension(name))
    if not match:
        return None
    year, month, day = match.groups()
    if not (1970 <= int(year) <= 2100):
        return None
    if not (1 <= int(month) <= 12 and 1 <= int(day) <= 31):
        return None
    return f"{year}-{month}-{day}"


def sanitize_table_name(stem: str, fallback: str = "unnamed") -> str:
    """Coerce an arbitrary filename stem into a legal BigQuery table id."""
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", stem).strip("_").lower()
    name = re.sub(r"_{2,}", "_", name)
    if not name:
        name = fallback
    if name[0].isdigit():
        name = f"t_{name}"
    return name[:1024]


def sanitize_column_name(raw: str, position: int) -> str:
    """Coerce a CSV header cell into a legal BigQuery column id."""
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", str(raw)).strip("_").lower()
    name = re.sub(r"_{2,}", "_", name)
    if not name:
        name = f"col_{position}"
    if name[0].isdigit() or name.startswith("_"):
        name = f"c_{name.lstrip('_')}"
    # BigQuery reserves the _TABLE_/_FILE_/_PARTITION prefixes.
    for reserved in ("_table_", "_file_", "_partition"):
        if name.startswith(reserved):
            name = f"c{name}"
    return name[:300]


# Wearable/health telemetry: enormous row counts, near-zero information per row,
# and nothing worth embedding. Matched against the family stem, anchored so
# `heart_rate_2026-04-05` matches but a document named "heart rate notes" does
# not get caught by accident.
HEALTH_FAMILY_PATTERNS = [
    r"^(daily_)?heart_rate(_variability|_zones)?$",
    r"^(daily_)?resting_heart_rate$",
    r"^time_in_heart_rate_zone$",
    r"^cardio_load.*$",
    r"^body_temperature$",
    r"^(daily_)?respiratory_rate.*$",
    r"^respiratory_rate_sleep_summary$",
    r"^oxygen_saturation$",
    r"^(minute_|daily_)?spo2$",
    r"^micro_(motion|stillness)$",
    r"^sedentary_period$",
    r"^continuous_eda$",
    r"^active_(minutes|zone_minutes|energy_burned)$",
    r"^calories$",
    r"^distance$",
    r"^steps$",
    r"^floors$",
    r"^altitude$",
    r"^swim_lengths_data$",
    r"^body_response_algorithm_features$",
    r"^(height|weight|bmi)$",
    r"^sleep(_score|_profile|_stages)?$",
    r"^stress.*$",
    r"^readiness.*$",
    r"^breathing_rate$",
    r"^exercise$",
    r"^menstrual.*$",
    r"^activity_goals$",
]

# Folders that are wholesale wearable exports.
HEALTH_PATH_PATTERNS = [
    r"(^|/)Fitbit(/|$)",
    r"(^|/)Google ?Fit(/|$)",
    r"(^|/)Apple ?Health(/|$)",
    r"(^|/)Health ?Data(/|$)",
]

_HEALTH_FAMILY_RE = re.compile("|".join(HEALTH_FAMILY_PATTERNS), re.IGNORECASE)
_HEALTH_PATH_RE = re.compile("|".join(HEALTH_PATH_PATTERNS), re.IGNORECASE)


def is_health(record: dict) -> bool:
    """True for wearable telemetry, by folder or by family name."""
    path = record.get("path") or ""
    if _HEALTH_PATH_RE.search(path):
        return True
    stem = record.get("family_stem")
    if stem and _HEALTH_FAMILY_RE.match(sanitize_table_name(stem)):
        return True
    return False


def partition(
    files: list[dict],
    skip_health: bool = False,
    exclude_path: str | None = None,
    exclude_family: str | None = None,
) -> tuple[list[dict], list[dict]]:
    """Split files into ``(kept, excluded)``.

    Excluded files are not dropped from the run -- the caller still records them
    in the manifest, marked as excluded with the reason. The census stays
    complete even when the load is scoped.
    """
    path_re = re.compile(exclude_path, re.IGNORECASE) if exclude_path else None
    family_re = re.compile(exclude_family, re.IGNORECASE) if exclude_family else None

    kept, excluded = [], []
    for record in files:
        reason = None
        if skip_health and is_health(record):
            reason = "health telemetry"
        elif path_re and path_re.search(record.get("path") or ""):
            reason = f"path matched {exclude_path!r}"
        elif family_re and family_re.search(record.get("family_stem") or ""):
            reason = f"family matched {exclude_family!r}"

        if reason:
            excluded.append({**record, "exclude_reason": reason})
        else:
            kept.append(record)
    return kept, excluded


@dataclass
class Family:
    """A set of Drive files that should land in one BigQuery table."""

    table: str
    stem: str
    fmt: str
    files: list[dict] = field(default_factory=list)

    @property
    def total_bytes(self) -> int:
        return sum(int(f.get("size_bytes") or 0) for f in self.files)


def build_families(files: list[dict]) -> dict[str, Family]:
    """Group classified tabular file records into families keyed by table name.

    Files sharing a stem but differing in format stay separate, because a CSV
    and an XLSX of the same stem rarely share a schema.
    """
    families: dict[str, Family] = {}
    for record in files:
        if record.get("kind") != "tabular":
            continue
        stem = family_stem(record["name"])
        fmt = record.get("fmt") or extension_of(record["name"])
        table = sanitize_table_name(stem)
        # Spreadsheets and CSVs of the same stem get distinct tables.
        if fmt in {"xlsx", "xls", "gsheet"}:
            table = f"{table}_sheet"
        elif fmt in {"json", "ndjson"}:
            table = f"{table}_json"
        key = table
        if key not in families:
            families[key] = Family(table=table, stem=stem, fmt=fmt)
        families[key].files.append(record)
    return families
__CLASSIFY_PY__'''

MODULES['drive.py'] = r'''__DRIVE_PY__
"""Drive enumeration and download.

Two modes, yielding identically shaped records so everything downstream is
mode-agnostic:

* ``walk`` / ``download`` -- the Drive API, for a service account or ADC.
* ``walk_local`` / ``read_local`` -- an already-mounted Drive (Colab's
  ``drive.mount``, or Drive for Desktop). Files are ordinary paths, so there is
  no pagination, no per-file API call, and no download quota. This is by far the
  cheaper mode for a Drive with thousands of files.
"""

from __future__ import annotations

import datetime as dt
import io
import json
import os
import time
from pathlib import Path

from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

from classify import FOLDER_MIME, classify, extension_of, family_stem, shard_date

FIELDS = (
    "nextPageToken,files(id,name,mimeType,size,md5Checksum,createdTime,"
    "modifiedTime,parents,trashed,shortcutDetails)"
)

# Google-native formats have no bytes to download; they must be exported.
EXPORT_MIMES = {
    "application/vnd.google-apps.spreadsheet": (
        "text/csv",
        "csv",
    ),
    "application/vnd.google-apps.document": (
        "text/plain",
        "txt",
    ),
    "application/vnd.google-apps.presentation": (
        "text/plain",
        "txt",
    ),
}

RETRYABLE_STATUS = {403, 429, 500, 502, 503, 504}


def _retry(call, attempts: int = 5, what: str = "drive call"):
    """Retry a Drive request through rate limits with exponential backoff."""
    delay = 2.0
    last: Exception | None = None
    for attempt in range(attempts):
        try:
            return call()
        except HttpError as exc:  # pragma: no cover - network dependent
            status = getattr(exc.resp, "status", None)
            if status not in RETRYABLE_STATUS or attempt == attempts - 1:
                raise
            last = exc
            time.sleep(delay)
            delay *= 2
    raise RuntimeError(f"{what} failed after {attempts} attempts: {last}")


def walk(service, root_id: str | None = None, include_trashed: bool = False):
    """Yield file records under ``root_id`` (whole My Drive when None).

    Breadth-first so that a partial run still covers whole folders, and so the
    path column can be built incrementally without a second lookup.
    """
    if root_id:
        root_meta = _retry(
            lambda: service.files()
            .get(fileId=root_id, fields="id,name,mimeType", supportsAllDrives=True)
            .execute(),
            what=f"get root {root_id}",
        )
        queue = [(root_id, root_meta.get("name", "/"))]
        seen_folders = {root_id}
    else:
        queue = [("root", "")]
        seen_folders = {"root"}

    while queue:
        folder_id, folder_path = queue.pop(0)
        page_token = None
        while True:
            query = f"'{folder_id}' in parents"
            if not include_trashed:
                query += " and trashed = false"

            def _list(token=page_token, q=query):
                return (
                    service.files()
                    .list(
                        q=q,
                        fields=FIELDS,
                        pageSize=1000,
                        pageToken=token,
                        supportsAllDrives=True,
                        includeItemsFromAllDrives=True,
                    )
                    .execute()
                )

            response = _retry(_list, what=f"list {folder_id}")

            for item in response.get("files", []):
                name = item.get("name", "")
                mime = item.get("mimeType", "")
                path = f"{folder_path}/{name}" if folder_path else name

                if mime == FOLDER_MIME:
                    if item["id"] not in seen_folders:
                        seen_folders.add(item["id"])
                        queue.append((item["id"], path))
                    continue

                # Shortcuts point elsewhere; the target is enumerated on its own.
                if mime == "application/vnd.google-apps.shortcut":
                    continue

                kind, fmt = classify(name, mime)
                yield {
                    "file_id": item["id"],
                    "name": name,
                    "path": path,
                    "mime_type": mime,
                    "extension": extension_of(name),
                    "size_bytes": int(item["size"]) if item.get("size") else None,
                    "md5_checksum": item.get("md5Checksum"),
                    "created_time": item.get("createdTime"),
                    "modified_time": item.get("modifiedTime"),
                    "parent_id": (item.get("parents") or [None])[0],
                    "kind": kind,
                    "fmt": fmt,
                    "family_stem": family_stem(name) if kind == "tabular" else None,
                    "shard_date": shard_date(name),
                }

            page_token = response.get("nextPageToken")
            if not page_token:
                break


# A mounted Drive represents Google-native files as small JSON stub files with
# these extensions. The stub holds the real file id but none of the data, so the
# bytes still have to be exported through the API.
STUB_EXTENSIONS = {
    "gsheet": "application/vnd.google-apps.spreadsheet",
    "gdoc": "application/vnd.google-apps.document",
    "gslides": "application/vnd.google-apps.presentation",
    "gdraw": "application/vnd.google-apps.drawing",
    "gform": "application/vnd.google-apps.form",
}

# Mount bookkeeping that is not user data.
SKIP_NAMES = {".shortcut-targets-by-id", ".file-revisions-by-id", ".Trash", ".DS_Store"}


def _stub_file_id(path: Path) -> str | None:
    """Pull the Drive file id out of a mounted .gsheet/.gdoc stub."""
    try:
        payload = json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except (OSError, json.JSONDecodeError):
        return None
    # Stubs carry either {"doc_id": ...} or {"url": "...open?id=FILE_ID"}.
    if isinstance(payload, dict):
        if payload.get("doc_id"):
            return str(payload["doc_id"])
        url = payload.get("url") or ""
        if "id=" in url:
            return url.split("id=", 1)[1].split("&", 1)[0]
    return None


def walk_local(root: str, include_hidden: bool = False):
    """Yield file records from a mounted Drive directory tree.

    Native Google files are surfaced with their real Drive id and MIME type so
    that ``read_local`` can export them through the API; everything else is read
    straight off disk.
    """
    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"{root_path} is not a directory")

    for dirpath, dirnames, filenames in os.walk(root_path):
        # Prune in place so os.walk does not descend into them.
        dirnames[:] = [
            d
            for d in dirnames
            if d not in SKIP_NAMES and (include_hidden or not d.startswith("."))
        ]
        for filename in filenames:
            if filename in SKIP_NAMES:
                continue
            if not include_hidden and filename.startswith("."):
                continue

            full = Path(dirpath) / filename
            try:
                stat = full.stat()
            except OSError:
                continue

            relative = full.relative_to(root_path).as_posix()
            ext = extension_of(filename)

            if ext in STUB_EXTENSIONS:
                mime = STUB_EXTENSIONS[ext]
                file_id = _stub_file_id(full) or f"local:{relative}"
                size = None  # the stub's size is meaningless
            else:
                mime = ""
                file_id = f"local:{relative}"
                size = stat.st_size

            kind, fmt = classify(filename, mime)
            yield {
                "file_id": file_id,
                "name": filename,
                "path": relative,
                "local_path": str(full),
                "mime_type": mime,
                "extension": ext,
                "size_bytes": size,
                "md5_checksum": None,
                "created_time": _iso(stat.st_ctime),
                "modified_time": _iso(stat.st_mtime),
                "parent_id": Path(dirpath).name,
                "kind": kind,
                "fmt": fmt,
                "family_stem": family_stem(filename) if kind == "tabular" else None,
                "shard_date": shard_date(filename),
            }


def _iso(timestamp: float) -> str:
    return dt.datetime.fromtimestamp(timestamp, dt.timezone.utc).isoformat()


def read_local(record: dict, service=None) -> tuple[bytes, str]:
    """Read a mounted file's bytes, exporting native Google files via the API.

    ``service`` is only needed for native files; plain files never touch the API.
    """
    mime = record.get("mime_type") or ""
    if mime in EXPORT_MIMES:
        if service is None:
            raise RuntimeError(
                f"{record['name']} is a native Google file and needs a Drive "
                "service to export; pass credentials or skip it"
            )
        return download(service, record["file_id"], mime)
    return Path(record["local_path"]).read_bytes(), ""


def download(service, file_id: str, mime_type: str) -> tuple[bytes, str]:
    """Fetch a file's bytes. Returns ``(data, effective_format)``.

    Google-native files are exported; everything else is downloaded verbatim.
    """
    if mime_type in EXPORT_MIMES:
        export_mime, fmt = EXPORT_MIMES[mime_type]
        request = service.files().export_media(fileId=file_id, mimeType=export_mime)
    else:
        fmt = ""
        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)

    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request, chunksize=8 * 1024 * 1024)
    done = False
    while not done:
        _, done = _retry(downloader.next_chunk, what=f"download {file_id}")
    return buffer.getvalue(), fmt
__DRIVE_PY__'''

MODULES['parse.py'] = r'''__PARSE_PY__
"""Turn downloaded bytes into DataFrames, and documents into text."""

from __future__ import annotations

import io
import json

import pandas as pd

from classify import sanitize_column_name

# Read everything as string first, then let BigQuery/pandas infer on the merged
# frame. Per-file inference is what causes schema drift across a family (one
# day's file has all-integer values, the next has a decimal).
_READ_KWARGS = {"dtype": str, "keep_default_na": False, "na_values": [""]}

MAX_TEXT_CHARS = 900_000  # keep a row comfortably under BigQuery's 100 MB cap


def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Sanitize headers and de-duplicate collisions."""
    seen: dict[str, int] = {}
    columns = []
    for position, raw in enumerate(frame.columns):
        name = sanitize_column_name(raw, position)
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        columns.append(name)
    frame.columns = columns
    return frame


def read_tabular(data: bytes, fmt: str, name: str) -> list[tuple[str, pd.DataFrame]]:
    """Parse bytes into ``[(sheet_suffix, frame)]``.

    Multi-sheet workbooks yield one entry per sheet; everything else yields one
    entry with an empty suffix.
    """
    if fmt in {"csv", "tsv"}:
        sep = "\t" if fmt == "tsv" else ","
        frame = _read_csv(data, sep)
        return [("", normalize_columns(frame))]

    if fmt in {"xlsx", "xls", "xlsm"}:
        engine = "openpyxl" if fmt != "xls" else "xlrd"
        book = pd.read_excel(
            io.BytesIO(data), sheet_name=None, engine=engine, **_READ_KWARGS
        )
        out = []
        for sheet_name, frame in book.items():
            if frame.empty:
                continue
            out.append((str(sheet_name), normalize_columns(frame)))
        return out

    if fmt in {"json", "ndjson"}:
        return [("", normalize_columns(_read_json(data)))]

    raise ValueError(f"unsupported tabular format {fmt!r} for {name!r}")


def _read_csv(data: bytes, sep: str) -> pd.DataFrame:
    """Read a CSV, tolerating the encodings Takeout exports show up in."""
    last: Exception | None = None
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            return pd.read_csv(
                io.BytesIO(data),
                sep=sep,
                encoding=encoding,
                engine="python",
                on_bad_lines="skip",
                **_READ_KWARGS,
            )
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            last = exc
    raise ValueError(f"could not parse CSV: {last}")


def _read_json(data: bytes) -> pd.DataFrame:
    """Flatten JSON into a frame, handling arrays, NDJSON, and wrapped objects."""
    text = data.decode("utf-8", errors="replace").strip()
    if not text:
        return pd.DataFrame()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        # Assume NDJSON.
        rows = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
        return pd.json_normalize(rows) if rows else pd.DataFrame()

    if isinstance(parsed, list):
        return pd.json_normalize(parsed)
    if isinstance(parsed, dict):
        # Takeout wraps the payload in a single key more often than not
        # (e.g. {"features": [...]} in Saved Places).
        list_values = [v for v in parsed.values() if isinstance(v, list)]
        if len(list_values) == 1:
            return pd.json_normalize(list_values[0])
        return pd.json_normalize([parsed])
    return pd.DataFrame({"value": [parsed]})


def extract_text(data: bytes, fmt: str) -> tuple[str, int, str]:
    """Return ``(text, page_count, method)`` for a document file."""
    if fmt in {"txt", "md", "html", "rtf"}:
        text = data.decode("utf-8", errors="replace")
        if fmt == "html":
            text = _strip_html(text)
        return text[:MAX_TEXT_CHARS], 0, f"decode:{fmt}"

    if fmt == "pdf":
        try:
            from pypdf import PdfReader
        except ImportError:
            return "", 0, "skipped:pypdf-missing"
        reader = PdfReader(io.BytesIO(data))
        pages = [(page.extract_text() or "") for page in reader.pages]
        return "\n".join(pages)[:MAX_TEXT_CHARS], len(pages), "pypdf"

    if fmt == "docx":
        try:
            import docx
        except ImportError:
            return "", 0, "skipped:python-docx-missing"
        document = docx.Document(io.BytesIO(data))
        text = "\n".join(p.text for p in document.paragraphs)
        return text[:MAX_TEXT_CHARS], 0, "python-docx"

    if fmt == "pptx":
        try:
            from pptx import Presentation
        except ImportError:
            return "", 0, "skipped:python-pptx-missing"
        deck = Presentation(io.BytesIO(data))
        chunks = []
        for slide in deck.slides:
            for shape in slide.shapes:
                if getattr(shape, "has_text_frame", False):
                    chunks.append(shape.text_frame.text)
        return "\n".join(chunks)[:MAX_TEXT_CHARS], len(deck.slides), "python-pptx"

    return "", 0, f"unsupported:{fmt}"


def _strip_html(text: str) -> str:
    import re

    text = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", text, flags=re.S | re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()


def reconcile(frames: list[pd.DataFrame]) -> pd.DataFrame:
    """Union frames with differing columns into one, preserving every column.

    Missing columns become NULL rather than dropping the row — a day's file that
    lacks a column should not silently lose its other fields.
    """
    if not frames:
        return pd.DataFrame()
    ordered: list[str] = []
    for frame in frames:
        for column in frame.columns:
            if column not in ordered:
                ordered.append(column)
    aligned = [frame.reindex(columns=ordered) for frame in frames]
    return pd.concat(aligned, ignore_index=True)
__PARSE_PY__'''

MODULES['chunk.py'] = r'''__CHUNK_PY__
"""Split extracted text into embedding-sized chunks.

Embedding models cap input length, so a 200-page PDF cannot become one vector.
It also should not: a single vector over a whole document averages away the
specific passage you were searching for. Chunking on natural boundaries with a
little overlap keeps each vector about one idea, and keeps a match pointing at a
findable location in the source.
"""

from __future__ import annotations

import re

# ~2000 chars is roughly 500 tokens: comfortably inside every current embedding
# model's window, and large enough that a chunk carries real context.
TARGET_CHARS = 2000
OVERLAP_CHARS = 200
MIN_CHARS = 60  # below this a chunk is noise (page numbers, stray headers)

# Split on blank lines first, then sentence ends, then hard-wrap. Preferring
# paragraph breaks keeps related sentences in the same vector.
_PARAGRAPH = re.compile(r"\n\s*\n")
_SENTENCE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(\"'])")
_WHITESPACE = re.compile(r"[ \t]+")


def normalize(text: str) -> str:
    """Collapse the whitespace damage that PDF extraction leaves behind."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\xa0", " ").replace("​", "")
    text = _WHITESPACE.sub(" ", text)
    # Three or more newlines carry no more meaning than two.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Join lines broken mid-sentence by hard-wrapping, but keep real breaks.
    text = re.sub(r"(?<=[a-z,;])\n(?=[a-z])", " ", text)
    return text.strip()


def split(text: str, target: int = TARGET_CHARS, overlap: int = OVERLAP_CHARS) -> list[str]:
    """Split text into overlapping chunks of about ``target`` characters."""
    text = normalize(text)
    if not text:
        return []
    if len(text) <= target:
        return [text]

    units = _units(text, target)

    chunks: list[str] = []
    current = ""
    for unit in units:
        if not current:
            current = unit
        elif len(current) + 1 + len(unit) <= target:
            current = f"{current}\n{unit}" if unit.startswith(("•", "-", "*")) else f"{current} {unit}"
        else:
            chunks.append(current)
            current = _tail(current, overlap) + " " + unit if overlap else unit
    if current:
        chunks.append(current)

    return [c.strip() for c in chunks if len(c.strip()) >= MIN_CHARS]


def _units(text: str, target: int) -> list[str]:
    """Break text into the largest pieces that still fit a chunk."""
    units: list[str] = []
    for paragraph in _PARAGRAPH.split(text):
        paragraph = paragraph.strip()
        if not paragraph:
            continue
        if len(paragraph) <= target:
            units.append(paragraph)
            continue
        # Too long: fall back to sentences.
        for sentence in _SENTENCE.split(paragraph):
            sentence = sentence.strip()
            if not sentence:
                continue
            if len(sentence) <= target:
                units.append(sentence)
            else:
                # Still too long (no punctuation at all -- transcripts, logs).
                units.extend(
                    sentence[i : i + target] for i in range(0, len(sentence), target)
                )
    return units


def _tail(text: str, overlap: int) -> str:
    """Last ``overlap`` chars of a chunk, trimmed to a word boundary."""
    if overlap <= 0 or len(text) <= overlap:
        return text
    tail = text[-overlap:]
    space = tail.find(" ")
    return tail[space + 1 :] if space != -1 else tail


def chunk_document(record: dict, text: str) -> list[dict]:
    """Turn one document's text into chunk rows ready for loading."""
    pieces = split(text)
    total = len(pieces)
    return [
        {
            "chunk_id": f"{record['file_id']}::{index:05d}",
            "file_id": record["file_id"],
            "name": record["name"],
            "path": record["path"],
            "chunk_index": index,
            "chunk_total": total,
            "char_count": len(piece),
            # The text column ML.GENERATE_EMBEDDING reads. Prefixing the source
            # name gives the vector a little document-level context, which
            # measurably helps retrieval on short chunks.
            "content": f"{record['name']}\n\n{piece}",
            "raw_content": piece,
        }
        for index, piece in enumerate(pieces)
    ]


def row_to_text(row: dict, table: str, max_chars: int = TARGET_CHARS) -> str:
    """Render a table row as prose so it can be embedded and searched.

    Telemetry rows are meaningless as text, which is why only non-telemetry
    tables get vectorized -- but a row from a spreadsheet of names, dates and
    notes is genuinely worth finding semantically.
    """
    parts = [f"Table: {table}"]
    for key, value in row.items():
        if key.startswith("_src_") or key == "_ingested_at":
            continue
        if value is None or value == "":
            continue
        text = str(value).strip()
        if not text or text.lower() in {"nan", "nat", "none"}:
            continue
        parts.append(f"{key.replace('_', ' ')}: {text}")
    return "\n".join(parts)[:max_chars]
__CHUNK_PY__'''

MODULES['dedupe.py'] = r'''__DEDUPE_PY__
"""Content-hash deduplication, run before anything expensive.

Measured on the target Drive: 100 spreadsheet files are 31 distinct documents.
`PGY-3.xlsx` exists 9 times; a 240 MB textbook PDF exists twice. Roughly a 3x
duplication factor.

That matters far more than it first looks:

* Embedding cost is paid per copy, so ~3x the bill for no extra information.
* Worse, the cross-link layer would be swamped. Every duplicate pair sits at
  cosine distance ~0, so `file_bridges` and `indirect_relations` would rank
  identical-file matches above every genuine connection. The expensive layer
  would produce confident noise.

So dedup runs first, and only canonical copies are parsed, chunked and embedded.
Nothing is lost: every copy stays in the manifest, marked `duplicate` with a
pointer to its canonical file, so "where are all the copies of this" remains
answerable.

The hashing is cheap because of one observation: two files of *different* sizes
cannot be identical. Only files whose size collides with another file are ever
read.
"""

from __future__ import annotations

import hashlib
import logging
from collections import defaultdict
from pathlib import Path

log = logging.getLogger(__name__)

READ_CHUNK = 1024 * 1024

# Hashing reads the whole file. Above this, hash a head+tail sample plus the
# size instead -- enough to separate genuinely different large files without
# reading gigabytes off a network mount.
FULL_HASH_LIMIT = 64 * 1024 * 1024
SAMPLE_BYTES = 4 * 1024 * 1024


def hash_local(path: str, size: int | None = None) -> str | None:
    """Content hash of a local file, sampling very large ones."""
    try:
        file_size = size if size is not None else Path(path).stat().st_size
        digest = hashlib.md5(usedforsecurity=False)
        digest.update(str(file_size).encode())
        with open(path, "rb") as handle:
            if file_size <= FULL_HASH_LIMIT:
                while chunk := handle.read(READ_CHUNK):
                    digest.update(chunk)
            else:
                digest.update(handle.read(SAMPLE_BYTES))
                handle.seek(-SAMPLE_BYTES, 2)
                digest.update(handle.read(SAMPLE_BYTES))
                digest.update(b"sampled")
        return digest.hexdigest()
    except OSError as exc:
        log.debug("cannot hash %s: %s", path, exc)
        return None


def assign_hashes(files: list[dict]) -> None:
    """Fill in `content_hash` on each record, in place.

    Drive's own `md5Checksum` is used when present (API mode gives it for free).
    Otherwise, only size-colliding local files are read -- a file with a unique
    size is already known to be unique.
    """
    by_size: dict[int, list[dict]] = defaultdict(list)
    for record in files:
        existing = record.get("md5_checksum")
        if existing:
            record["content_hash"] = existing
            continue
        record["content_hash"] = None
        size = record.get("size_bytes")
        if size:
            by_size[int(size)].append(record)

    candidates = [r for group in by_size.values() if len(group) > 1 for r in group]
    if not candidates:
        return
    log.info(
        "hashing %d of %d files (only sizes that collide can be duplicates)",
        len(candidates),
        len(files),
    )
    for record in candidates:
        path = record.get("local_path")
        if path:
            record["content_hash"] = hash_local(path, record.get("size_bytes"))


def partition_duplicates(files: list[dict]) -> tuple[list[dict], list[dict]]:
    """Split into ``(canonical, duplicates)`` by content hash.

    The canonical copy is the shallowest path, then the shortest, then the
    lexicographically first -- a stable rule that tends to pick the copy in the
    most sensible place rather than one buried in a nested backup tree.
    """
    groups: dict[str, list[dict]] = defaultdict(list)
    unhashed: list[dict] = []
    for record in files:
        digest = record.get("content_hash")
        if digest:
            groups[digest].append(record)
        else:
            unhashed.append(record)

    canonical: list[dict] = list(unhashed)
    duplicates: list[dict] = []
    for digest, group in groups.items():
        if len(group) == 1:
            canonical.append(group[0])
            continue
        group.sort(key=lambda r: (
            (r.get("path") or "").count("/"),
            len(r.get("path") or ""),
            r.get("path") or "",
        ))
        keeper, rest = group[0], group[1:]
        canonical.append(keeper)
        for record in rest:
            duplicates.append({
                **record,
                "duplicate_of": keeper["file_id"],
                "duplicate_of_path": keeper.get("path"),
            })
    return canonical, duplicates


def summarize(canonical: list[dict], duplicates: list[dict]) -> dict:
    wasted = sum(int(r.get("size_bytes") or 0) for r in duplicates)
    return {
        "total": len(canonical) + len(duplicates),
        "canonical": len(canonical),
        "duplicates": len(duplicates),
        "wasted_bytes": wasted,
        "factor": round((len(canonical) + len(duplicates)) / max(len(canonical), 1), 2),
    }


def top_duplicated(duplicates: list[dict], limit: int = 15) -> list[tuple[str, int, int]]:
    """``(name, copies, wasted_bytes)`` for the worst offenders."""
    counts: dict[str, list[dict]] = defaultdict(list)
    for record in duplicates:
        counts[record["name"]].append(record)
    rows = [
        (name, len(group) + 1, sum(int(r.get("size_bytes") or 0) for r in group))
        for name, group in counts.items()
    ]
    rows.sort(key=lambda row: -row[2])
    return rows[:limit]
__DEDUPE_PY__'''

MODULES['quickinsights.py'] = r'''__QUICKINSIGHTS_PY__
"""Insights that need no embeddings, no Vertex connection, and no LLM.

The expensive path (embed -> cross-link -> extract entities) is genuinely more
powerful, but it is also slow, costs real money, and cannot run until a Vertex
connection exists. A large share of useful insight does not need any of it, and
making people climb through the expensive layer to see anything is the wrong
order of work.

Everything here is plain SQL over `drive_raw.file_manifest` and
`drive_documents`. It runs in seconds, costs cents, and depends on nothing that
can be misconfigured.

The one non-obvious piece is `term_bridges`: it crosses documents against each
other using shared *rare terms* rather than vectors. Less semantically subtle
than embeddings -- it will not spot a paraphrase -- but for records work it is
arguably better, because what actually matters is shared identifiers: a name, a
case number, a specific phrase. It needs no model at all.
"""

from __future__ import annotations

INSIGHTS_DATASET = "drive_insights"

# A term appearing in this fraction of documents or more is structural
# boilerplate ("patient", "the", a letterhead) and links everything to
# everything, so it is excluded from bridging.
MAX_DOC_FRACTION = 0.20

# Below this many characters a "document" is a stub and its terms are noise.
MIN_DOC_CHARS = 200


def duplicates_view_sql(project: str) -> str:
    """Every duplicate group, ranked by wasted space."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.duplicates` AS
SELECT
  ANY_VALUE(name)                    AS name,
  content_hash,
  COUNT(*)                           AS copies,
  ANY_VALUE(size_bytes)              AS bytes_each,
  (COUNT(*) - 1) * ANY_VALUE(size_bytes) AS bytes_wasted,
  ARRAY_AGG(path ORDER BY path)      AS paths,
  ARRAY_AGG(DISTINCT REGEXP_EXTRACT(path, r'^([^/]+)') IGNORE NULLS
            ORDER BY REGEXP_EXTRACT(path, r'^([^/]+)')) AS top_folders
FROM `{project}.drive_raw.file_manifest`
WHERE content_hash IS NOT NULL
GROUP BY content_hash
HAVING copies > 1
""".strip()


def storage_view_sql(project: str) -> str:
    """Where the bytes actually are, and how much of it is redundant."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.storage` AS
WITH per_folder AS (
  SELECT
    IFNULL(REGEXP_EXTRACT(path, r'^([^/]+)'), '(root)') AS top_folder,
    kind,
    COUNT(*)        AS files,
    SUM(size_bytes) AS bytes,
    COUNTIF(ingest_status = 'duplicate') AS duplicate_files,
    SUM(IF(ingest_status = 'duplicate', size_bytes, 0)) AS duplicate_bytes
  FROM `{project}.drive_raw.file_manifest`
  GROUP BY top_folder, kind
)
SELECT
  top_folder, kind, files, bytes,
  duplicate_files, duplicate_bytes,
  SAFE_DIVIDE(duplicate_bytes, bytes) AS redundant_fraction
FROM per_folder
""".strip()


def census_view_sql(project: str) -> str:
    """The straight answer to "what is actually in here"."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.census` AS
SELECT
  kind,
  fmt,
  COUNT(*)                             AS files,
  COUNT(DISTINCT content_hash)         AS distinct_contents,
  SUM(size_bytes)                      AS bytes,
  MIN(DATE(modified_time))             AS earliest,
  MAX(DATE(modified_time))             AS latest,
  COUNTIF(ingest_status = 'loaded')    AS loaded,
  COUNTIF(ingest_status = 'duplicate') AS duplicates,
  COUNTIF(ingest_status = 'excluded')  AS excluded,
  COUNTIF(ingest_status = 'failed')    AS failed
FROM `{project}.drive_raw.file_manifest`
GROUP BY kind, fmt
""".strip()


def name_clusters_view_sql(project: str) -> str:
    """Files whose names differ only by a copy/version marker.

    Catches the near-duplicates that content hashing misses: `report.pdf` and
    `report (1).pdf` with one byte changed are two distinct contents but one
    document, and a `_v2`/`final`/`copy` family is usually a version chain worth
    collapsing by hand.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.name_clusters` AS
WITH normalised AS (
  SELECT
    file_id, name, path, size_bytes, content_hash, modified_time,
    -- Strip copy markers, version tags and trailing numbers from the stem.
    TRIM(REGEXP_REPLACE(
      REGEXP_REPLACE(
        LOWER(REGEXP_REPLACE(name, r'\\.[^.]+$', '')),
        r'\\s*\\(\\d+\\)|[ _-]+(copy|final|latest|v\\d+|new|old)$', ''
      ), r'[^a-z0-9]+', ' '
    )) AS stem
  FROM `{project}.drive_raw.file_manifest`
  WHERE kind IN ('document', 'tabular')
)
SELECT
  stem,
  COUNT(*)                     AS variants,
  COUNT(DISTINCT content_hash) AS distinct_contents,
  SUM(size_bytes)              AS bytes,
  MIN(DATE(modified_time))     AS first_modified,
  MAX(DATE(modified_time))     AS last_modified,
  ARRAY_AGG(STRUCT(name, path, size_bytes) ORDER BY path LIMIT 20) AS files
FROM normalised
WHERE stem != ''
GROUP BY stem
HAVING variants > 1
""".strip()


def term_index_sql(project: str) -> str:
    """Rare-term index over document text. No model involved.

    Terms are lowercased alphanumeric tokens of 4+ characters. Document
    frequency is computed so boilerplate can be excluded: a term present in a
    fifth of all documents links everything to everything and carries no signal.
    """
    return f"""
CREATE OR REPLACE TABLE `{project}.{INSIGHTS_DATASET}.doc_terms`
CLUSTER BY term
AS
WITH docs AS (
  SELECT file_id, name, path, content
  FROM `{project}.drive_documents.documents`
  WHERE content IS NOT NULL AND LENGTH(content) >= {MIN_DOC_CHARS}
),
total AS (SELECT COUNT(*) AS n FROM docs),
tokens AS (
  SELECT
    d.file_id, d.name, d.path,
    token
  FROM docs d,
  UNNEST(REGEXP_EXTRACT_ALL(LOWER(d.content), r'[a-z][a-z0-9]{{3,}}')) AS token
),
counted AS (
  SELECT
    file_id, name, path, token AS term,
    COUNT(*) AS term_count
  FROM tokens
  GROUP BY file_id, name, path, term
),
doc_freq AS (
  SELECT term, COUNT(DISTINCT file_id) AS docs_with_term
  FROM counted GROUP BY term
)
SELECT
  c.file_id, c.name, c.path, c.term, c.term_count,
  f.docs_with_term,
  f.docs_with_term / (SELECT n FROM total) AS doc_fraction
FROM counted c
JOIN doc_freq f USING (term)
WHERE f.docs_with_term BETWEEN 2 AND CAST(
        (SELECT n FROM total) * {MAX_DOC_FRACTION} AS INT64)
  AND LENGTH(c.term) >= 4
""".strip()


def term_bridges_view_sql(project: str, min_shared: int = 4) -> str:
    """Document pairs joined by shared rare terms -- crossing without vectors.

    Scored by summed inverse document frequency, so a pair sharing one very rare
    term (a case number, an unusual surname) outranks a pair sharing several
    merely uncommon ones.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.term_bridges` AS
WITH pairs AS (
  SELECT
    a.file_id AS a_file_id, b.file_id AS b_file_id,
    ANY_VALUE(a.name) AS a_name, ANY_VALUE(b.name) AS b_name,
    ANY_VALUE(a.path) AS a_path, ANY_VALUE(b.path) AS b_path,
    COUNT(*) AS shared_terms,
    -- Inverse document frequency: rarer shared terms are worth more.
    ROUND(SUM(1.0 / a.docs_with_term), 3) AS score,
    ARRAY_AGG(a.term ORDER BY a.docs_with_term LIMIT 12) AS rarest_shared
  FROM `{project}.{INSIGHTS_DATASET}.doc_terms` a
  JOIN `{project}.{INSIGHTS_DATASET}.doc_terms` b
    ON a.term = b.term AND a.file_id < b.file_id
  GROUP BY a_file_id, b_file_id
)
SELECT
  a_name, b_name, a_path, b_path, shared_terms, score, rarest_shared,
  REGEXP_EXTRACT(a_path, r'^([^/]+)') != REGEXP_EXTRACT(b_path, r'^([^/]+)')
    AS crosses_folder
FROM pairs
WHERE shared_terms >= {min_shared}
""".strip()


def distinctive_terms_view_sql(project: str) -> str:
    """What each document is *about*, by its rarest frequent terms.

    A crude but effective topic label, and a fast way to scan a corpus you have
    not read.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.distinctive_terms` AS
SELECT
  name, path,
  ARRAY_AGG(term ORDER BY term_count / docs_with_term DESC LIMIT 12) AS terms
FROM `{project}.{INSIGHTS_DATASET}.doc_terms`
GROUP BY name, path
""".strip()


def timeline_view_sql(project: str) -> str:
    """Corpus activity over time, from file dates alone."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.file_timeline` AS
SELECT
  DATE_TRUNC(DATE(modified_time), MONTH) AS month,
  IFNULL(REGEXP_EXTRACT(path, r'^([^/]+)'), '(root)') AS top_folder,
  kind,
  COUNT(*)        AS files,
  SUM(size_bytes) AS bytes
FROM `{project}.drive_raw.file_manifest`
WHERE modified_time IS NOT NULL
GROUP BY month, top_folder, kind
""".strip()


def headline_sql(project: str) -> str:
    """One row of the numbers worth seeing first."""
    return f"""
SELECT
  COUNT(*)                                        AS files,
  COUNT(DISTINCT content_hash)                    AS distinct_contents,
  COUNTIF(ingest_status = 'duplicate')            AS duplicate_files,
  ROUND(SUM(IF(ingest_status = 'duplicate', size_bytes, 0)) / 1073741824, 2)
                                                  AS duplicate_gib,
  ROUND(SUM(size_bytes) / 1073741824, 2)          AS total_gib,
  COUNTIF(kind = 'document')                      AS documents,
  COUNTIF(kind = 'tabular')                       AS tabular,
  COUNTIF(kind = 'media')                         AS media,
  COUNTIF(ingest_status = 'excluded')             AS excluded,
  COUNTIF(ingest_status = 'failed')               AS failed,
  MIN(DATE(modified_time))                        AS earliest,
  MAX(DATE(modified_time))                        AS latest
FROM `{project}.drive_raw.file_manifest`
""".strip()


def all_views(project: str) -> list[tuple[str, str]]:
    """Views in dependency order. `doc_terms` is a table others read."""
    return [
        ("census", census_view_sql(project)),
        ("duplicates", duplicates_view_sql(project)),
        ("storage", storage_view_sql(project)),
        ("name_clusters", name_clusters_view_sql(project)),
        ("file_timeline", timeline_view_sql(project)),
        ("doc_terms", term_index_sql(project)),
        ("term_bridges", term_bridges_view_sql(project)),
        ("distinctive_terms", distinctive_terms_view_sql(project)),
    ]
__QUICKINSIGHTS_PY__'''

MODULES['vectorize.py'] = r'''__VECTORIZE_PY__
"""Embed loaded content into a searchable vector table in BigQuery.

Embeddings are generated *inside* BigQuery via `ML.GENERATE_EMBEDDING` against a
remote Vertex AI model, so the text never leaves BigQuery and there is no
client-side embedding loop to babysit or pay egress on.

Four steps, each idempotent:

1. a CLOUD_RESOURCE connection from BigQuery to Vertex AI  (one-time, needs the
   `bq` CLI and an IAM grant -- see `connection_commands`)
2. a remote MODEL wrapping a Vertex text-embedding endpoint
3. `drive_vectors.embeddings` -- one row per embedded unit, all sources unified
4. a VECTOR INDEX over it, plus a `search` table function

Everything lands in ONE vector table on purpose. A document chunk, a spreadsheet
row and a photo's filename are all just text with provenance, and one index over
all of them means one query searches the whole Drive.
"""

from __future__ import annotations

import json
import logging
import shutil
import subprocess

log = logging.getLogger(__name__)

VECTORS_DATASET = "drive_vectors"
EMBEDDINGS_TABLE = "embeddings"
MODEL_NAME = "embedder"
CONNECTION_NAME = "drive_vertex"
INDEX_NAME = "embeddings_idx"

# Vertex text-embedding endpoint. `text-embedding-005` is the stable default;
# `gemini-embedding-001` is newer and higher quality but returns 3072 dims,
# which makes for a much larger index. Override with --embedding-model.
DEFAULT_ENDPOINT = "text-embedding-005"

# BigQuery only *uses* a vector index once the table is large enough; below this
# it silently falls back to a brute-force scan, which is correct but slower.
INDEX_MIN_ROWS = 5000

# ML.GENERATE_EMBEDDING is billed per input token and rate limited, so text is
# embedded in batches rather than one enormous query.
EMBED_BATCH_ROWS = 20_000


def connection_commands(project: str, location: str) -> list[str]:
    """Shell commands that create the Vertex connection and grant it access.

    Kept as text rather than executed blindly: the IAM grant widens a project's
    permissions, so it should be visible and reviewable before it runs.
    """
    conn = f"{project}.{location}.{CONNECTION_NAME}"
    return [
        f"bq mk --connection --location={location} --project_id={project} "
        f"--connection_type=CLOUD_RESOURCE {CONNECTION_NAME}",
        f"bq show --format=json --connection {conn}",
        # The connection gets its own service account; it needs Vertex access.
        f"gcloud projects add-iam-policy-binding {project} "
        f"--member=serviceAccount:$CONNECTION_SA --role=roles/aiplatform.user",
    ]


def ensure_connection(project: str, location: str, dry_run: bool = False) -> str | None:
    """Create the connection if absent and return its service account id."""
    if not shutil.which("bq"):
        log.warning("bq CLI not found; create the connection manually")
        return None

    conn = f"{project}.{location}.{CONNECTION_NAME}"
    show = ["bq", "show", "--format=json", "--connection", conn]

    result = subprocess.run(show, capture_output=True, text=True)
    if result.returncode != 0:
        if dry_run:
            log.info("[dry-run] would create connection %s", conn)
            return None
        log.info("creating connection %s", conn)
        create = subprocess.run(
            [
                "bq", "mk", "--connection",
                f"--location={location}",
                f"--project_id={project}",
                "--connection_type=CLOUD_RESOURCE",
                CONNECTION_NAME,
            ],
            capture_output=True,
            text=True,
        )
        if create.returncode != 0:
            raise RuntimeError(f"could not create connection: {create.stderr.strip()}")
        result = subprocess.run(show, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"connection created but unreadable: {result.stderr.strip()}")

    try:
        payload = json.loads(result.stdout)
    except json.JSONDecodeError:
        return None
    account = (payload.get("cloudResource") or {}).get("serviceAccountId")
    if account:
        log.info("connection service account: %s", account)
        log.info(
            "if embedding fails with a permission error, grant it Vertex access:\n"
            "  gcloud projects add-iam-policy-binding %s \\\n"
            "    --member=serviceAccount:%s --role=roles/aiplatform.user",
            project,
            account,
        )
    return account


def create_model_sql(project: str, location: str, endpoint: str = DEFAULT_ENDPOINT) -> str:
    return f"""
CREATE OR REPLACE MODEL `{project}.{VECTORS_DATASET}.{MODEL_NAME}`
REMOTE WITH CONNECTION `{project}.{location}.{CONNECTION_NAME}`
OPTIONS (ENDPOINT = '{endpoint}')
""".strip()


def create_embeddings_table_sql(project: str) -> str:
    """The unified vector table.

    `embedding` is left as ARRAY<FLOAT64> rather than a fixed-width type because
    the dimension depends on which endpoint is configured.
    """
    return f"""
CREATE TABLE IF NOT EXISTS `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}` (
  vector_id     STRING NOT NULL,
  source_kind   STRING,      -- document_chunk | table_row | file_metadata
  source_table  STRING,
  file_id       STRING,
  name          STRING,
  path          STRING,
  chunk_index   INT64,
  chunk_total   INT64,
  content       STRING,
  embedding     ARRAY<FLOAT64>,
  embedded_at   TIMESTAMP
)
""".strip()


def create_staging_table_sql(project: str) -> str:
    """Text waiting to be embedded. Emptied as batches succeed."""
    return f"""
CREATE TABLE IF NOT EXISTS `{project}.{VECTORS_DATASET}.embed_queue` (
  vector_id     STRING NOT NULL,
  source_kind   STRING,
  source_table  STRING,
  file_id       STRING,
  name          STRING,
  path          STRING,
  chunk_index   INT64,
  chunk_total   INT64,
  content       STRING,
  queued_at     TIMESTAMP
)
""".strip()


def enqueue_from_chunks_sql(project: str) -> str:
    """Queue every document chunk not already embedded."""
    return f"""
INSERT INTO `{project}.{VECTORS_DATASET}.embed_queue`
  (vector_id, source_kind, source_table, file_id, name, path,
   chunk_index, chunk_total, content, queued_at)
SELECT
  c.chunk_id                        AS vector_id,
  'document_chunk'                  AS source_kind,
  'drive_documents.document_chunks' AS source_table,
  c.file_id, c.name, c.path, c.chunk_index, c.chunk_total,
  c.content,
  CURRENT_TIMESTAMP()               AS queued_at
FROM `{project}.drive_documents.document_chunks` c
WHERE c.content IS NOT NULL AND LENGTH(TRIM(c.content)) > 0
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}` e
    WHERE e.vector_id = c.chunk_id
  )
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{VECTORS_DATASET}.embed_queue` q
    WHERE q.vector_id = c.chunk_id
  )
""".strip()


def enqueue_file_metadata_sql(project: str) -> str:
    """Queue a text line per non-tabular file so media is findable by name.

    A photo has no extractable content, but its name and folder path carry real
    signal -- this is what makes "that scan from the hospital" locatable at all.
    """
    return f"""
INSERT INTO `{project}.{VECTORS_DATASET}.embed_queue`
  (vector_id, source_kind, source_table, file_id, name, path,
   chunk_index, chunk_total, content, queued_at)
SELECT
  CONCAT('meta::', m.file_id) AS vector_id,
  'file_metadata'             AS source_kind,
  'drive_raw.file_manifest'   AS source_table,
  m.file_id, m.name, m.path,
  CAST(NULL AS INT64) AS chunk_index,
  CAST(NULL AS INT64) AS chunk_total,
  CONCAT(
    'File: ', m.name, '\\n',
    'Folder: ', IFNULL(REGEXP_EXTRACT(m.path, r'^(.*)/[^/]+$'), '(root)'), '\\n',
    'Type: ', IFNULL(m.fmt, 'unknown'), ' (', IFNULL(m.kind, 'unknown'), ')', '\\n',
    'Modified: ', IFNULL(CAST(DATE(m.modified_time) AS STRING), 'unknown')
  ) AS content,
  CURRENT_TIMESTAMP() AS queued_at
FROM `{project}.drive_raw.file_manifest` m
WHERE m.kind IN ('media', 'other')
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}` e
    WHERE e.vector_id = CONCAT('meta::', m.file_id)
  )
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{VECTORS_DATASET}.embed_queue` q
    WHERE q.vector_id = CONCAT('meta::', m.file_id)
  )
""".strip()


def enqueue_table_rows_sql(project: str, table: str, limit: int) -> str:
    """Queue a text rendering of each row of one table.

    Only worth doing for tables whose rows are descriptive rather than
    telemetry, which is why the caller chooses which tables. Rows are rendered
    as JSON so column names travel with the values -- `"grade":"fail"` embeds
    far better than a bare `fail`.
    """
    return f"""
INSERT INTO `{project}.{VECTORS_DATASET}.embed_queue`
  (vector_id, source_kind, source_table, file_id, name, path,
   chunk_index, chunk_total, content, queued_at)
SELECT
  CONCAT('row::{table}::', CAST(rn AS STRING)) AS vector_id,
  'table_row'                                  AS source_kind,
  'drive_tables.{table}'                       AS source_table,
  file_id,
  name,
  CAST(NULL AS STRING)                         AS path,
  CAST(NULL AS INT64)                          AS chunk_index,
  CAST(NULL AS INT64)                          AS chunk_total,
  CONCAT('Table: {table}\\n', content)          AS content,
  CURRENT_TIMESTAMP()                          AS queued_at
FROM (
  SELECT
    ROW_NUMBER() OVER (ORDER BY _src_file_name, _src_date) AS rn,
    _src_file_id      AS file_id,
    _src_file_name    AS name,
    TO_JSON_STRING(t) AS content
  FROM `{project}.drive_tables.{table}` t
  LIMIT {limit}
)
WHERE content IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}` e
    WHERE e.vector_id = CONCAT('row::{table}::', CAST(rn AS STRING))
  )
""".strip()


def embed_batch_sql(
    project: str, batch_rows: int = EMBED_BATCH_ROWS, task_type: str = "RETRIEVAL_DOCUMENT"
) -> str:
    """Embed one batch off the queue and move it into the vector table.

    `flatten_json_output` gives `ml_generate_embedding_result` as a plain
    ARRAY<FLOAT64>. Rows whose status is non-empty failed and are left in the
    queue rather than being written with a null vector.
    """
    return f"""
INSERT INTO `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}`
  (vector_id, source_kind, source_table, file_id, name, path,
   chunk_index, chunk_total, content, embedding, embedded_at)
SELECT
  vector_id, source_kind, source_table, file_id, name, path,
  chunk_index, chunk_total, content,
  ml_generate_embedding_result AS embedding,
  CURRENT_TIMESTAMP()          AS embedded_at
FROM ML.GENERATE_EMBEDDING(
  MODEL `{project}.{VECTORS_DATASET}.{MODEL_NAME}`,
  (
    SELECT vector_id, source_kind, source_table, file_id, name, path,
           chunk_index, chunk_total, content
    FROM `{project}.{VECTORS_DATASET}.embed_queue`
    LIMIT {batch_rows}
  ),
  STRUCT(TRUE AS flatten_json_output, '{task_type}' AS task_type)
)
WHERE ml_generate_embedding_status = ''
  AND ARRAY_LENGTH(ml_generate_embedding_result) > 0
""".strip()


def dequeue_embedded_sql(project: str) -> str:
    """Drop queue rows that made it into the vector table."""
    return f"""
DELETE FROM `{project}.{VECTORS_DATASET}.embed_queue` q
WHERE EXISTS (
  SELECT 1 FROM `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}` e
  WHERE e.vector_id = q.vector_id
)
""".strip()


def queue_depth_sql(project: str) -> str:
    return f"SELECT COUNT(*) AS n FROM `{project}.{VECTORS_DATASET}.embed_queue`"


def create_index_sql(project: str) -> str:
    return f"""
CREATE OR REPLACE VECTOR INDEX {INDEX_NAME}
ON `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}`(embedding)
OPTIONS (index_type = 'IVF', distance_type = 'COSINE')
""".strip()


def create_search_function_sql(project: str) -> str:
    """A table function so searching is one short query, not a nested mess."""
    return f"""
CREATE OR REPLACE TABLE FUNCTION `{project}.{VECTORS_DATASET}.search`(
  query STRING, top_k INT64
)
AS (
  SELECT
    base.name        AS name,
    base.path        AS path,
    base.source_kind AS source_kind,
    base.chunk_index AS chunk_index,
    distance,
    base.content     AS content
  FROM VECTOR_SEARCH(
    TABLE `{project}.{VECTORS_DATASET}.{EMBEDDINGS_TABLE}`,
    'embedding',
    (
      SELECT ml_generate_embedding_result AS embedding
      FROM ML.GENERATE_EMBEDDING(
        MODEL `{project}.{VECTORS_DATASET}.{MODEL_NAME}`,
        (SELECT query AS content),
        STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
      )
    ),
    top_k => top_k,
    distance_type => 'COSINE'
  )
);
""".strip()
__VECTORIZE_PY__'''

MODULES['graph.py'] = r'''__GRAPH_PY__
"""Cross the data against itself, then cross the crossings.

Vectors alone give you similarity search: ask a question, get passages. That is
still a *static* index -- it only answers what you thought to ask.

This layer makes the corpus reason about itself, in two passes:

  Pass 1 (crossing).  Every chunk is searched against every other chunk and the
      nearest neighbours in OTHER files are recorded as links. A passage in a
      probation letter and a row in a residency evaluation that talk about the
      same thing become an explicit edge, with no query typed by anyone.

  Pass 2 (crossing the crossings).  Entities are extracted from each chunk, then
      pushed *through* those edges. Two people who never appear in the same file
      but are each central to files that link to one another are surfaced as
      related. Timelines assemble across sources. Bridges -- single links that
      join otherwise separate clusters -- are ranked, because in practice a
      bridge is where the interesting thing is.

Everything is a view or an incremental table refreshed on a schedule, so the
picture moves as the Drive moves rather than being a snapshot.

The LLM-dependent statements are deliberately confined to `extract_entities_sql`
and `insight_feed_sql`. BigQuery's generative SQL surface moves faster than the
rest of it, so if a signature has drifted the fix is in one place.
"""

from __future__ import annotations

GRAPH_DATASET = "drive_graph"
INSIGHTS_DATASET = "drive_insights"

MENTIONS_TABLE = "entity_mentions"
ENTITIES_TABLE = "entities"
LINKS_TABLE = "cross_links"
EXTRACTOR_MODEL = "extractor"
STATE_TABLE = "pipeline_state"

# Gemini endpoint for extraction and summarisation. Rotates faster than
# anything else here; override with --text-model.
DEFAULT_TEXT_ENDPOINT = "gemini-2.5-flash"

# Neighbours recorded per chunk. Beyond ~8 the tail is noise and the link table
# grows quadratically for nothing.
NEIGHBOURS_PER_CHUNK = 8

# Cosine distance above which a "link" is not really a link. Tuned
# conservatively: 0.35 keeps genuine topical overlap and drops boilerplate.
MAX_LINK_DISTANCE = 0.35

ENTITY_TYPES = ("PERSON", "ORG", "DATE", "CASE_NUMBER", "LOCATION", "MEDICAL", "MONEY", "TOPIC")


def sql_literal(text: str) -> str:
    """Escape text for a single-quoted BigQuery string literal.

    Prompts are prose and prose contains apostrophes; one unescaped `'` in a
    prompt silently truncates the SQL statement into something that either fails
    or, worse, parses as something else.
    """
    return text.replace("\\", "\\\\").replace("'", "\\'")


# --------------------------------------------------------------- pass 1: links


def create_links_table_sql(project: str) -> str:
    return f"""
CREATE TABLE IF NOT EXISTS `{project}.{GRAPH_DATASET}.{LINKS_TABLE}` (
  link_id      STRING NOT NULL,
  a_vector_id  STRING,
  b_vector_id  STRING,
  a_file_id    STRING,
  b_file_id    STRING,
  a_name       STRING,
  b_name       STRING,
  a_path       STRING,
  b_path       STRING,
  a_kind       STRING,
  b_kind       STRING,
  a_excerpt    STRING,
  b_excerpt    STRING,
  distance     FLOAT64,
  linked_at    TIMESTAMP
)
PARTITION BY DATE(linked_at)
CLUSTER BY a_file_id, b_file_id
""".strip()


def build_links_sql(
    project: str,
    vectors_dataset: str,
    embeddings_table: str,
    neighbours: int = NEIGHBOURS_PER_CHUNK,
    max_distance: float = MAX_LINK_DISTANCE,
) -> str:
    """Search the embedding table against itself and keep cross-file neighbours.

    Two constraints make this useful rather than noise:

    * `a_file_id != b_file_id` -- neighbours within one document are just the
      document being locally coherent, which we already knew.
    * an ordered pair guard, so A-B and B-A collapse to one edge.
    """
    return f"""
CREATE OR REPLACE TABLE `{project}.{GRAPH_DATASET}.{LINKS_TABLE}`
PARTITION BY DATE(linked_at)
CLUSTER BY a_file_id, b_file_id
AS
WITH neighbours AS (
  SELECT
    query.vector_id   AS a_vector_id,
    base.vector_id    AS b_vector_id,
    query.file_id     AS a_file_id,
    base.file_id      AS b_file_id,
    query.name        AS a_name,
    base.name         AS b_name,
    query.path        AS a_path,
    base.path         AS b_path,
    query.source_kind AS a_kind,
    base.source_kind  AS b_kind,
    SUBSTR(query.content, 1, 400) AS a_excerpt,
    SUBSTR(base.content,  1, 400) AS b_excerpt,
    distance
  FROM VECTOR_SEARCH(
    TABLE `{project}.{vectors_dataset}.{embeddings_table}`, 'embedding',
    TABLE `{project}.{vectors_dataset}.{embeddings_table}`,
    query_column_to_search => 'embedding',
    top_k => {neighbours + 1},
    distance_type => 'COSINE'
  )
  WHERE query.file_id != base.file_id
    AND distance <= {max_distance}
)
SELECT
  TO_HEX(MD5(CONCAT(
    LEAST(a_vector_id, b_vector_id), '|', GREATEST(a_vector_id, b_vector_id)
  ))) AS link_id,
  ANY_VALUE(a_vector_id) AS a_vector_id,
  ANY_VALUE(b_vector_id) AS b_vector_id,
  ANY_VALUE(a_file_id)   AS a_file_id,
  ANY_VALUE(b_file_id)   AS b_file_id,
  ANY_VALUE(a_name)      AS a_name,
  ANY_VALUE(b_name)      AS b_name,
  ANY_VALUE(a_path)      AS a_path,
  ANY_VALUE(b_path)      AS b_path,
  ANY_VALUE(a_kind)      AS a_kind,
  ANY_VALUE(b_kind)      AS b_kind,
  ANY_VALUE(a_excerpt)   AS a_excerpt,
  ANY_VALUE(b_excerpt)   AS b_excerpt,
  MIN(distance)          AS distance,
  CURRENT_TIMESTAMP()    AS linked_at
FROM neighbours
GROUP BY link_id
""".strip()


# ------------------------------------------------------------ pass 2: entities


def create_extractor_model_sql(
    project: str, vectors_dataset: str, location: str, connection: str, endpoint: str
) -> str:
    return f"""
CREATE OR REPLACE MODEL `{project}.{vectors_dataset}.{EXTRACTOR_MODEL}`
REMOTE WITH CONNECTION `{project}.{location}.{connection}`
OPTIONS (ENDPOINT = '{endpoint}')
""".strip()


def create_mentions_table_sql(project: str) -> str:
    return f"""
CREATE TABLE IF NOT EXISTS `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` (
  mention_id   STRING NOT NULL,
  vector_id    STRING,
  file_id      STRING,
  name         STRING,
  path         STRING,
  source_kind  STRING,
  entity_text  STRING,
  entity_type  STRING,
  entity_norm  STRING,
  extracted_at TIMESTAMP
)
CLUSTER BY entity_norm, entity_type
""".strip()


def extract_entities_sql(
    project: str, vectors_dataset: str, batch_rows: int = 2000
) -> str:
    """Pull entities out of chunks with Gemini, one batch at a time.

    The prompt demands strict JSON and the result is parsed with SAFE.PARSE_JSON,
    so a malformed response yields no rows for that chunk instead of failing the
    statement. Chunks already extracted are skipped, which makes this resumable
    and cheap to re-run as new files arrive.

    NOTE: this is one of two statements using BigQuery's generative SQL surface
    and it has not been executed against a live project. If `ML.GENERATE_TEXT`
    has drifted, this function and `insight_feed_sql` are the only places to fix.
    """
    prompt = sql_literal(
        "Extract named entities from the text. Return ONLY minified JSON, with no "
        "markdown fence, shaped exactly like this: "
        '{"entities":[{"text":"","type":""}]} '
        "where type is one of " + "|".join(ENTITY_TYPES) + ". "
        "Copy the exact surface form from the text into the text field. "
        "Omit generic words, pronouns, and anything uncertain. "
        "Return at most 25 entities. Text:"
    ) + "\\n\\n"
    return f"""
INSERT INTO `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}`
  (mention_id, vector_id, file_id, name, path, source_kind,
   entity_text, entity_type, entity_norm, extracted_at)
WITH generated AS (
  SELECT
    vector_id, file_id, name, path, source_kind,
    SAFE.PARSE_JSON(
      REGEXP_REPLACE(ml_generate_text_llm_result, r'^```(?:json)?|```$', '')
    ) AS payload
  FROM ML.GENERATE_TEXT(
    MODEL `{project}.{vectors_dataset}.{EXTRACTOR_MODEL}`,
    (
      SELECT
        e.vector_id, e.file_id, e.name, e.path, e.source_kind,
        CONCAT('{prompt}', SUBSTR(e.content, 1, 6000)) AS prompt
      FROM `{project}.{vectors_dataset}.embeddings` e
      WHERE e.source_kind IN ('document_chunk', 'table_row')
        AND NOT EXISTS (
          SELECT 1 FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` m
          WHERE m.vector_id = e.vector_id
        )
      LIMIT {batch_rows}
    ),
    STRUCT(0.0 AS temperature, 1024 AS max_output_tokens,
           TRUE AS flatten_json_output)
  )
)
flattened AS (
  SELECT
    g.vector_id, g.file_id, g.name, g.path, g.source_kind,
    JSON_VALUE(entity, '$.text')        AS entity_text,
    UPPER(JSON_VALUE(entity, '$.type')) AS entity_type
  FROM generated g
  CROSS JOIN UNNEST(JSON_QUERY_ARRAY(g.payload, '$.entities')) AS entity
  WHERE g.payload IS NOT NULL
)
SELECT
  TO_HEX(MD5(CONCAT(vector_id, '|', entity_text, '|', entity_type))) AS mention_id,
  vector_id, file_id, name, path, source_kind,
  entity_text,
  entity_type,
  -- Join key: casefolded, punctuation dropped, whitespace runs collapsed, so
  -- "Dr. Rahman" and "dr rahman" resolve to the same entity.
  TRIM(REGEXP_REPLACE(LOWER(entity_text), r'[^a-z0-9]+', ' ')) AS entity_norm
  , CURRENT_TIMESTAMP() AS extracted_at
FROM flattened
WHERE entity_text IS NOT NULL
  AND LENGTH(TRIM(entity_text)) BETWEEN 2 AND 200
  AND entity_type IN ({", ".join(f"'{t}'" for t in ENTITY_TYPES)})
""".strip()


def pending_extraction_sql(project: str, vectors_dataset: str) -> str:
    return f"""
SELECT COUNT(*) AS n
FROM `{project}.{vectors_dataset}.embeddings` e
WHERE e.source_kind IN ('document_chunk', 'table_row')
  AND NOT EXISTS (
    SELECT 1 FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` m
    WHERE m.vector_id = e.vector_id
  )
""".strip()


def build_entities_sql(project: str) -> str:
    """Roll mentions up into resolved entities.

    `entity_norm` is the join key, so "Dr. Rahman" and "dr rahman" collapse. The
    most frequent surface form becomes the display name.
    """
    return f"""
CREATE OR REPLACE TABLE `{project}.{GRAPH_DATASET}.{ENTITIES_TABLE}`
CLUSTER BY entity_type, entity_norm
AS
WITH ranked_forms AS (
  SELECT
    entity_norm, entity_type, entity_text,
    COUNT(*) AS form_count,
    ROW_NUMBER() OVER (
      PARTITION BY entity_norm, entity_type ORDER BY COUNT(*) DESC, entity_text
    ) AS rn
  FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}`
  WHERE entity_norm != ''
  GROUP BY entity_norm, entity_type, entity_text
)
SELECT
  m.entity_norm,
  m.entity_type,
  ANY_VALUE(f.entity_text)          AS display_name,
  COUNT(*)                          AS mention_count,
  COUNT(DISTINCT m.file_id)         AS file_count,
  COUNT(DISTINCT m.source_kind)     AS kind_count,
  ARRAY_AGG(DISTINCT m.name IGNORE NULLS ORDER BY m.name LIMIT 25) AS files,
  CURRENT_TIMESTAMP()               AS refreshed_at
FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` m
JOIN ranked_forms f
  ON f.entity_norm = m.entity_norm AND f.entity_type = m.entity_type AND f.rn = 1
WHERE m.entity_norm != ''
GROUP BY m.entity_norm, m.entity_type
""".strip()


# ---------------------------------------------------------------- the insights


def entity_timeline_view_sql(project: str) -> str:
    """Every dated appearance of an entity, from any source.

    Dates come from three places: a shard date on the source file, a DATE-typed
    entity extracted from the same chunk, and the file's modified time as a last
    resort. Crossing an entity against time across sources is what turns a pile
    of files into a chronology.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.entity_timeline` AS
WITH
-- One date per chunk, not one row per DATE entity in it. Joining the mentions
-- table to itself unaggregated would multiply every mention by its chunk's date
-- count.
chunk_dates AS (
  SELECT vector_id, MIN(SAFE.PARSE_DATE('%Y-%m-%d', entity_text)) AS in_text_date
  FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}`
  WHERE entity_type = 'DATE'
  GROUP BY vector_id
)
SELECT
  m.entity_norm, m.entity_type, m.entity_text,
  COALESCE(
    cd.in_text_date,
    SAFE_CAST(man.shard_date AS DATE),
    DATE(man.modified_time)
  ) AS event_date,
  CASE
    WHEN cd.in_text_date IS NOT NULL  THEN 'in_text'
    WHEN man.shard_date IS NOT NULL   THEN 'filename'
    ELSE 'file_mtime'
  END AS date_source,
  m.name AS file_name, m.path, m.source_kind, m.file_id, m.vector_id
FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` m
LEFT JOIN `{project}.drive_raw.file_manifest` man ON man.file_id = m.file_id
LEFT JOIN chunk_dates cd ON cd.vector_id = m.vector_id
WHERE m.entity_type != 'DATE'
  AND COALESCE(cd.in_text_date, SAFE_CAST(man.shard_date AS DATE),
               DATE(man.modified_time)) IS NOT NULL
""".strip()


def cooccurrence_view_sql(project: str) -> str:
    """Entities that appear in the same chunk -- the direct crossing."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.entity_cooccurrence` AS
SELECT
  a.entity_norm  AS entity_a,
  b.entity_norm  AS entity_b,
  a.entity_type  AS type_a,
  b.entity_type  AS type_b,
  COUNT(*)                        AS together_count,
  COUNT(DISTINCT a.file_id)       AS file_count,
  ARRAY_AGG(DISTINCT a.name IGNORE NULLS ORDER BY a.name LIMIT 10) AS files
FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` a
JOIN `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` b
  ON a.vector_id = b.vector_id
 AND a.entity_norm < b.entity_norm   -- one row per unordered pair
WHERE a.entity_norm != '' AND b.entity_norm != ''
GROUP BY entity_a, entity_b, type_a, type_b
HAVING together_count >= 2
""".strip()


def indirect_relations_view_sql(project: str) -> str:
    """Crossing the crossings.

    Entities that never share a chunk, but sit at either end of a semantic link
    between two different files. This is the one view that can tell you something
    no single document contains -- the relationship exists only in the geometry
    between documents.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.indirect_relations` AS
WITH bridged AS (
  SELECT
    LEAST(ma.entity_norm, mb.entity_norm)    AS entity_a,
    GREATEST(ma.entity_norm, mb.entity_norm) AS entity_b,
    ma.entity_type AS type_a,
    mb.entity_type AS type_b,
    l.a_name, l.b_name, l.distance
  FROM `{project}.{GRAPH_DATASET}.{LINKS_TABLE}` l
  JOIN `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` ma ON ma.vector_id = l.a_vector_id
  JOIN `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` mb ON mb.vector_id = l.b_vector_id
  WHERE ma.entity_norm != '' AND mb.entity_norm != ''
    AND ma.entity_norm != mb.entity_norm
),
-- Materialise the co-occurring pairs once. Doing this as a correlated NOT EXISTS
-- would re-run a mentions self-join per candidate row.
direct AS (
  SELECT DISTINCT
    LEAST(x.entity_norm, y.entity_norm)    AS entity_a,
    GREATEST(x.entity_norm, y.entity_norm) AS entity_b
  FROM `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` x
  JOIN `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` y
    ON x.vector_id = y.vector_id AND x.entity_norm != y.entity_norm
  WHERE x.entity_norm != '' AND y.entity_norm != ''
)
SELECT
  b.entity_a, b.entity_b, b.type_a, b.type_b,
  COUNT(*)          AS bridge_count,
  MIN(b.distance)   AS closest_distance,
  ARRAY_AGG(DISTINCT b.a_name IGNORE NULLS ORDER BY b.a_name LIMIT 5) AS from_files,
  ARRAY_AGG(DISTINCT b.b_name IGNORE NULLS ORDER BY b.b_name LIMIT 5) AS to_files
FROM bridged b
LEFT JOIN direct d
  ON d.entity_a = b.entity_a AND d.entity_b = b.entity_b
-- Only *indirect*: the pair must never share a chunk anywhere.
WHERE d.entity_a IS NULL
GROUP BY b.entity_a, b.entity_b, b.type_a, b.type_b
HAVING bridge_count >= 2
""".strip()


def file_bridges_view_sql(project: str) -> str:
    """File pairs joined by many links -- where two parts of the Drive meet.

    Ranked by link count and closeness. A high-scoring pair of files from
    unrelated folders is usually the most interesting thing in the corpus.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.file_bridges` AS
SELECT
  a_name, b_name,
  ANY_VALUE(a_path) AS a_path,
  ANY_VALUE(b_path) AS b_path,
  ANY_VALUE(a_kind) AS a_kind,
  ANY_VALUE(b_kind) AS b_kind,
  COUNT(*)      AS link_count,
  MIN(distance) AS closest_distance,
  AVG(distance) AS mean_distance,
  -- Different top-level folders means the link crosses a boundary in how the
  -- Drive is organised, which is where a surprise usually lives.
  REGEXP_EXTRACT(ANY_VALUE(a_path), r'^([^/]+)') !=
  REGEXP_EXTRACT(ANY_VALUE(b_path), r'^([^/]+)') AS crosses_folder,
  ARRAY_AGG(STRUCT(a_excerpt, b_excerpt, distance)
            ORDER BY distance LIMIT 3) AS examples
FROM `{project}.{GRAPH_DATASET}.{LINKS_TABLE}`
GROUP BY a_name, b_name
HAVING link_count >= 2
""".strip()


def entity_gaps_view_sql(project: str) -> str:
    """Entities that show up in one kind of source but not another.

    A person named all over your documents but absent from every spreadsheet, or
    the reverse, is either a data gap or a finding. Either way it is worth seeing.
    """
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.entity_gaps` AS
SELECT
  e.entity_norm, e.entity_type, e.display_name,
  e.mention_count, e.file_count,
  COUNTIF(m.source_kind = 'document_chunk') AS in_documents,
  COUNTIF(m.source_kind = 'table_row')      AS in_tables,
  COUNTIF(m.source_kind = 'file_metadata')  AS in_filenames,
  CASE
    WHEN COUNTIF(m.source_kind = 'table_row') = 0
     AND COUNTIF(m.source_kind = 'document_chunk') > 0 THEN 'documents_only'
    WHEN COUNTIF(m.source_kind = 'document_chunk') = 0
     AND COUNTIF(m.source_kind = 'table_row') > 0      THEN 'tables_only'
    ELSE 'both'
  END AS coverage
FROM `{project}.{GRAPH_DATASET}.{ENTITIES_TABLE}` e
JOIN `{project}.{GRAPH_DATASET}.{MENTIONS_TABLE}` m
  ON m.entity_norm = e.entity_norm AND m.entity_type = e.entity_type
GROUP BY e.entity_norm, e.entity_type, e.display_name, e.mention_count, e.file_count
""".strip()


def activity_view_sql(project: str) -> str:
    """What changed lately, across every dataset. The 'is it alive' view."""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.recent_activity` AS
SELECT
  DATE(modified_time) AS day,
  kind,
  COUNT(*)            AS files,
  SUM(size_bytes)     AS bytes,
  COUNTIF(ingest_status = 'loaded')   AS loaded,
  COUNTIF(ingest_status = 'failed')   AS failed,
  COUNTIF(ingest_status = 'excluded') AS excluded
FROM `{project}.drive_raw.file_manifest`
WHERE modified_time IS NOT NULL
GROUP BY day, kind
""".strip()


def insight_feed_sql(project: str, vectors_dataset: str, top_n: int = 40) -> str:
    """Have Gemini write up the strongest cross-file bridges in plain language.

    This is the readable end of the pipeline: rather than making you interpret
    cosine distances, each bridge gets a sentence on what the two passages share
    and whether it looks meaningful.

    NOTE: the second of the two unverified generative statements. See
    `extract_entities_sql`.
    """
    instruction = sql_literal(
        "Two passages from different files in the same Drive were matched as "
        "semantically similar. In at most two sentences, say concretely what they "
        "share, then end with the single word MEANINGFUL or COINCIDENTAL. Be "
        "blunt: most matches are coincidental boilerplate and should be called "
        "that rather than dressed up."
    )
    return f"""
CREATE OR REPLACE TABLE `{project}.{INSIGHTS_DATASET}.insight_feed`
AS
SELECT
  a_name, b_name, distance, crosses_folder,
  ml_generate_text_llm_result AS assessment,
  REGEXP_CONTAINS(UPPER(ml_generate_text_llm_result), r'MEANINGFUL') AS looks_meaningful,
  CURRENT_TIMESTAMP() AS generated_at
FROM ML.GENERATE_TEXT(
  MODEL `{project}.{vectors_dataset}.{EXTRACTOR_MODEL}`,
  (
    SELECT
      a_name, b_name, closest_distance AS distance, crosses_folder,
      CONCAT(
        '{instruction}',
        '\\n\\nFILE A (', a_name, '):\\n', examples[OFFSET(0)].a_excerpt,
        '\\n\\nFILE B (', b_name, '):\\n', examples[OFFSET(0)].b_excerpt
      ) AS prompt
    FROM `{project}.{INSIGHTS_DATASET}.file_bridges`
    WHERE ARRAY_LENGTH(examples) > 0
    ORDER BY crosses_folder DESC, link_count DESC, closest_distance
    LIMIT {top_n}
  ),
  STRUCT(0.2 AS temperature, 256 AS max_output_tokens, TRUE AS flatten_json_output)
)
""".strip()


def ask_function_sql(project: str, vectors_dataset: str) -> str:
    """Retrieval-augmented question answering over the whole Drive, in one call.

    Semantic search fetches the passages; Gemini answers from them and is told to
    cite filenames and to admit when the corpus does not contain the answer.
    """
    return f"""
CREATE OR REPLACE TABLE FUNCTION `{project}.{INSIGHTS_DATASET}.ask`(
  question STRING, passages INT64
)
AS (
  WITH hits AS (
    SELECT base.name AS name, base.content AS content, distance
    FROM VECTOR_SEARCH(
      TABLE `{project}.{vectors_dataset}.embeddings`, 'embedding',
      (
        SELECT ml_generate_embedding_result AS embedding
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{project}.{vectors_dataset}.embedder`,
          (SELECT question AS content),
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
      ),
      top_k => passages, distance_type => 'COSINE'
    )
  ),
  context AS (
    SELECT STRING_AGG(
      CONCAT('--- ', name, ' ---\\n', SUBSTR(content, 1, 2000)), '\\n\\n'
      ORDER BY distance
    ) AS body
    FROM hits
  )
  SELECT
    question,
    ml_generate_text_llm_result AS answer,
    (SELECT ARRAY_AGG(DISTINCT name) FROM hits) AS sources
  FROM ML.GENERATE_TEXT(
    MODEL `{project}.{vectors_dataset}.{EXTRACTOR_MODEL}`,
    (
      SELECT CONCAT(
        'Answer the question using only the excerpts below. Cite the filenames ',
        'you relied on. If the excerpts do not contain the answer, say so ',
        'plainly rather than guessing.\\n\\nQUESTION: ', question,
        '\\n\\nEXCERPTS:\\n', body
      ) AS prompt
      FROM context
    ),
    STRUCT(0.2 AS temperature, 1024 AS max_output_tokens, TRUE AS flatten_json_output)
  )
);
""".strip()


# -------------------------------------------------------------------- keep it live


def create_state_table_sql(project: str) -> str:
    """Watermarks, so refreshes do incremental work instead of full rebuilds."""
    return f"""
CREATE TABLE IF NOT EXISTS `{project}.{GRAPH_DATASET}.{STATE_TABLE}` (
  stage       STRING NOT NULL,
  watermark   TIMESTAMP,
  rows_seen   INT64,
  note        STRING,
  updated_at  TIMESTAMP
)
""".strip()


def record_state_sql(project: str, stage: str, rows: int, note: str = "") -> str:
    safe_note = note.replace("'", "''")
    return f"""
MERGE `{project}.{GRAPH_DATASET}.{STATE_TABLE}` T
USING (SELECT '{stage}' AS stage) S
ON T.stage = S.stage
WHEN MATCHED THEN UPDATE SET
  watermark = CURRENT_TIMESTAMP(), rows_seen = {rows},
  note = '{safe_note}', updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (stage, watermark, rows_seen, note, updated_at)
  VALUES ('{stage}', CURRENT_TIMESTAMP(), {rows}, '{safe_note}', CURRENT_TIMESTAMP())
""".strip()


def health_view_sql(project: str) -> str:
    """One view answering 'is this thing actually current?'"""
    return f"""
CREATE OR REPLACE VIEW `{project}.{INSIGHTS_DATASET}.pipeline_health` AS
SELECT
  stage,
  watermark            AS last_run,
  rows_seen,
  note,
  TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), watermark, HOUR) AS hours_since,
  TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), watermark, HOUR) > 48 AS stale
FROM `{project}.{GRAPH_DATASET}.{STATE_TABLE}`
""".strip()


def schedule_commands(project: str, location: str) -> list[tuple[str, str]]:
    """`bq query --schedule` commands that keep the derived layer refreshing.

    The ingest step still has to run somewhere with Drive access; everything
    downstream of the vector table is pure SQL and BigQuery can drive it on a
    timer with no machine of yours involved.
    """
    def cmd(name: str, schedule: str, sql: str) -> tuple[str, str]:
        one_line = " ".join(sql.split())
        return (
            name,
            f"bq query --use_legacy_sql=false --project_id={project} "
            f"--location={location} --schedule='{schedule}' "
            f"--display_name='drive:{name}' "
            f'"{one_line}"',
        )

    return [
        cmd("links", "every 24 hours", build_links_sql(project, "drive_vectors", "embeddings")),
        cmd("entities", "every 24 hours", build_entities_sql(project)),
        cmd("insight_feed", "every 24 hours", insight_feed_sql(project, "drive_vectors")),
    ]
__GRAPH_PY__'''

MODULES['preflight.py'] = r'''__PREFLIGHT_PY__
"""Fail early, with the fix in the message.

The first real run of this pipeline hits a handful of environment problems that
all produce the same unhelpful shape: a Google API traceback ending in a 403 or
404 whose text does not say what to do. Each check here turns one of those into a
sentence naming the exact remedy.

Checks are ordered cheapest-first and every one is read-only.
"""

from __future__ import annotations

import logging

from google.api_core import exceptions as gexc

log = logging.getLogger(__name__)

CONSOLE = "https://console.cloud.google.com"


class PreflightError(RuntimeError):
    """A problem the user has to fix before the run can work."""


def check_project(client, project: str) -> None:
    """Confirm the BigQuery API is enabled and the project is reachable.

    A project that has never used BigQuery returns 403 with "has not been used in
    project ... before or it is disabled", which reads like a permissions problem
    and is not one.
    """
    try:
        list(client.list_datasets(project=project, max_results=1))
    except gexc.Forbidden as exc:
        message = str(exc)
        if "has not been used" in message or "is disabled" in message:
            raise PreflightError(
                f"The BigQuery API is not enabled on project {project!r}.\n"
                f"  Enable it:  gcloud services enable bigquery.googleapis.com "
                f"--project {project}\n"
                f"  or visit:   {CONSOLE}/apis/library/bigquery.googleapis.com"
                f"?project={project}"
            ) from exc
        raise PreflightError(
            f"No BigQuery access to project {project!r}. The authenticated "
            f"account needs roles/bigquery.dataEditor and roles/bigquery.jobUser.\n"
            f"  Details: {message.splitlines()[0]}"
        ) from exc
    except gexc.NotFound as exc:
        raise PreflightError(
            f"Project {project!r} not found. Check the id (not the display name) "
            f"at {CONSOLE}/home/dashboard"
        ) from exc


def check_dataset_locations(client, project: str, datasets: list[str], location: str) -> None:
    """Catch a dataset that already exists in a different region.

    BigQuery cannot join across locations, and `create_dataset` on an existing
    dataset raises Conflict, which the loader swallows. The mismatch then surfaces
    much later as a confusing "not found in location" on a query.
    """
    wrong: list[tuple[str, str]] = []
    for dataset_id in datasets:
        try:
            existing = client.get_dataset(f"{project}.{dataset_id}")
        except gexc.NotFound:
            continue
        except gexc.Forbidden:
            continue
        if existing.location and existing.location.upper() != location.upper():
            wrong.append((dataset_id, existing.location))

    if wrong:
        listed = "\n".join(f"    {name} is in {loc}" for name, loc in wrong)
        raise PreflightError(
            f"Dataset location mismatch. This run uses {location!r}, but:\n"
            f"{listed}\n"
            "  BigQuery cannot query across locations. Either re-run with "
            f"--location {wrong[0][1]}, or delete those datasets and let this "
            "run recreate them."
        )


def check_write_access(client, project: str, location: str) -> None:
    """Confirm the account can actually create a dataset, not just read.

    Read access is common and write access is not; discovering the difference
    after a long enumeration wastes the whole walk.
    """
    from google.cloud import bigquery

    probe_id = f"{project}.drive_preflight_probe"
    dataset = bigquery.Dataset(probe_id)
    dataset.location = location
    try:
        client.create_dataset(dataset)
    except gexc.Conflict:
        pass  # left over from an earlier run; that is proof enough
    except gexc.Forbidden as exc:
        raise PreflightError(
            f"The authenticated account cannot create datasets in {project!r}.\n"
            "  It needs roles/bigquery.dataEditor (to write) and "
            "roles/bigquery.jobUser (to run load jobs).\n"
            f"  Grant at: {CONSOLE}/iam-admin/iam?project={project}"
        ) from exc
    else:
        try:
            client.delete_dataset(probe_id, not_found_ok=True)
        except gexc.GoogleAPIError:
            log.debug("could not clean up the preflight probe dataset")


def check_vertex_connection(client, project: str, location: str, connection: str) -> None:
    """Confirm the Vertex connection and its IAM grant look usable.

    Only checked before the embedding stages, since everything before them works
    without Vertex.
    """
    import shutil
    import subprocess

    if not shutil.which("bq"):
        log.warning(
            "bq CLI not on PATH, so the Vertex connection cannot be verified. "
            "If embedding fails, create it with:\n"
            "  bq mk --connection --location=%s --project_id=%s "
            "--connection_type=CLOUD_RESOURCE %s",
            location, project, connection,
        )
        return

    result = subprocess.run(
        ["bq", "show", "--format=json", "--connection",
         f"{project}.{location}.{connection}"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise PreflightError(
            f"No Vertex connection {project}.{location}.{connection}.\n"
            f"  Create it:  bq mk --connection --location={location} "
            f"--project_id={project} --connection_type=CLOUD_RESOURCE {connection}\n"
            "  Then grant its service account roles/aiplatform.user "
            "(the notebook's step 7 does both)."
        )


def run(loader, project: str, datasets: list[str], location: str,
        need_vertex: bool = False, connection: str = "drive_vertex") -> None:
    """Run every applicable check. Raises PreflightError with the remedy."""
    if loader.dry_run:
        log.info("[dry-run] skipping preflight")
        return

    client = loader.client
    log.info("preflight: checking project, permissions and dataset locations ...")
    check_project(client, project)
    check_write_access(client, project, location)
    check_dataset_locations(client, project, datasets, location)
    if need_vertex:
        check_vertex_connection(client, project, location, connection)
    log.info("preflight: ok")
__PREFLIGHT_PY__'''

MODULES['bq.py'] = r'''__BQ_PY__
"""BigQuery dataset/table management and loading."""

from __future__ import annotations

import io
import logging

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.api_core import exceptions as gexc
from google.cloud import bigquery

log = logging.getLogger(__name__)

MANIFEST_TABLE = "file_manifest"
DOCUMENTS_TABLE = "documents"

MANIFEST_SCHEMA = [
    bigquery.SchemaField("file_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("path", "STRING"),
    bigquery.SchemaField("mime_type", "STRING"),
    bigquery.SchemaField("extension", "STRING"),
    bigquery.SchemaField("size_bytes", "INT64"),
    bigquery.SchemaField("md5_checksum", "STRING"),
    bigquery.SchemaField("created_time", "TIMESTAMP"),
    bigquery.SchemaField("modified_time", "TIMESTAMP"),
    bigquery.SchemaField("parent_id", "STRING"),
    bigquery.SchemaField("kind", "STRING"),
    bigquery.SchemaField("fmt", "STRING"),
    bigquery.SchemaField("family_stem", "STRING"),
    bigquery.SchemaField("shard_date", "DATE"),
    bigquery.SchemaField("content_hash", "STRING"),
    bigquery.SchemaField("duplicate_of", "STRING"),
    bigquery.SchemaField("target_dataset", "STRING"),
    bigquery.SchemaField("target_table", "STRING"),
    bigquery.SchemaField("ingest_status", "STRING"),
    bigquery.SchemaField("ingest_error", "STRING"),
    bigquery.SchemaField("row_count", "INT64"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

CHUNKS_TABLE = "document_chunks"

CHUNKS_SCHEMA = [
    bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("file_id", "STRING"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("path", "STRING"),
    bigquery.SchemaField("chunk_index", "INT64"),
    bigquery.SchemaField("chunk_total", "INT64"),
    bigquery.SchemaField("char_count", "INT64"),
    # `content` is what gets embedded (name-prefixed); `raw_content` is the
    # verbatim passage, for display without the synthetic header.
    bigquery.SchemaField("content", "STRING"),
    bigquery.SchemaField("raw_content", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

DOCUMENTS_SCHEMA = [
    bigquery.SchemaField("file_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("path", "STRING"),
    bigquery.SchemaField("mime_type", "STRING"),
    bigquery.SchemaField("fmt", "STRING"),
    bigquery.SchemaField("size_bytes", "INT64"),
    bigquery.SchemaField("created_time", "TIMESTAMP"),
    bigquery.SchemaField("modified_time", "TIMESTAMP"),
    bigquery.SchemaField("page_count", "INT64"),
    bigquery.SchemaField("char_count", "INT64"),
    bigquery.SchemaField("extraction_method", "STRING"),
    bigquery.SchemaField("content", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

# Provenance columns prepended to every table in the tables dataset, so a row
# can always be traced back to the Drive file it came from.
PROVENANCE = ("_src_file_id", "_src_file_name", "_src_date", "_src_sheet", "_ingested_at")


# BigQuery field type -> the exact Arrow type its Parquet column must carry.
_ARROW_TYPES = {
    "STRING": pa.string(),
    "BYTES": pa.binary(),
    "INT64": pa.int64(),
    "INTEGER": pa.int64(),
    "FLOAT64": pa.float64(),
    "FLOAT": pa.float64(),
    "NUMERIC": pa.float64(),
    "BOOL": pa.bool_(),
    "BOOLEAN": pa.bool_(),
    "TIMESTAMP": pa.timestamp("us", tz="UTC"),
    "DATETIME": pa.timestamp("us"),
    "DATE": pa.date32(),
}


def to_parquet_bytes(frame: pd.DataFrame, schema: list) -> io.BytesIO:
    """Serialise a frame to Parquet with types pinned by the BigQuery schema.

    Casting pandas dtypes alone is not enough: a column that is entirely null
    serialises as Arrow `null` whatever its pandas dtype, and BigQuery rejects
    that for anything but a STRING field. Building the Arrow schema explicitly
    pins every column, empty or not.
    """
    frame = coerce_to_schema(frame, schema)
    fields = []
    for field in schema:
        arrow_type = _ARROW_TYPES.get(field.field_type.upper(), pa.string())
        if field.name not in frame.columns:
            # A schema field the caller never populated still needs a column, or
            # from_pandas rejects the frame.
            frame[field.name] = None
        fields.append(pa.field(field.name, arrow_type, nullable=field.mode != "REQUIRED"))

    table = pa.Table.from_pandas(
        frame[[f.name for f in schema]], schema=pa.schema(fields), preserve_index=False
    )
    buffer = io.BytesIO()
    pq.write_table(table, buffer)
    buffer.seek(0)
    return buffer


def coerce_to_schema(frame: pd.DataFrame, schema: list) -> pd.DataFrame:
    """Cast a DataFrame so its Parquet types match a declared BigQuery schema.

    Loading Parquet against an explicit schema is strict: the Parquet physical
    type has to match the declared field type. Two pandas behaviours break that
    silently, and both bite the very first load:

    * An INT64 column containing any None becomes float64, so Parquet carries
      `double` where BigQuery expects an integer.
    * ISO-8601 strings stay strings. A TIMESTAMP or DATE field receives
      `large_string` and the load is rejected.

    An all-null column is a third case: it serialises as Parquet `null`, which a
    STRING field tolerates but an INT64 or TIMESTAMP field does not, so the cast
    is applied even when there is nothing to convert.
    """
    frame = frame.copy()
    for field in schema:
        name, kind = field.name, field.field_type.upper()
        if name not in frame.columns:
            continue
        column = frame[name]
        try:
            if kind in {"INT64", "INTEGER"}:
                # Nullable integer, so None survives without forcing float.
                frame[name] = pd.to_numeric(column, errors="coerce").astype("Int64")
            elif kind in {"FLOAT64", "FLOAT"}:
                frame[name] = pd.to_numeric(column, errors="coerce").astype("Float64")
            elif kind in {"BOOL", "BOOLEAN"}:
                frame[name] = column.astype("boolean")
            elif kind == "TIMESTAMP":
                frame[name] = pd.to_datetime(column, errors="coerce", utc=True, format="ISO8601")
            elif kind == "DATE":
                parsed = pd.to_datetime(column, errors="coerce", format="ISO8601")
                # BigQuery wants a date, not a midnight timestamp.
                frame[name] = parsed.dt.date.astype("object").where(parsed.notna(), None)
            else:
                frame[name] = column.astype("string")
        except (TypeError, ValueError) as exc:
            raise RuntimeError(
                f"column {name!r} cannot be cast to {kind} for load: {exc}"
            ) from exc
    return frame


class Loader:
    def __init__(self, project: str, location: str = "US", dry_run: bool = False):
        self.project = project
        self.location = location
        self.dry_run = dry_run
        self.client = None if dry_run else bigquery.Client(project=project)

    # ---------------------------------------------------------------- datasets

    def ensure_dataset(self, dataset_id: str) -> None:
        if self.dry_run:
            log.info("[dry-run] ensure dataset %s.%s", self.project, dataset_id)
            return
        ref = bigquery.Dataset(f"{self.project}.{dataset_id}")
        ref.location = self.location
        try:
            self.client.create_dataset(ref)
            log.info("created dataset %s", dataset_id)
        except gexc.Conflict:
            log.debug("dataset %s already exists", dataset_id)

    def ensure_table(self, dataset_id: str, table_id: str, schema: list) -> None:
        if self.dry_run:
            log.info("[dry-run] ensure table %s.%s", dataset_id, table_id)
            return
        table = bigquery.Table(f"{self.project}.{dataset_id}.{table_id}", schema=schema)
        try:
            self.client.create_table(table)
            log.info("created table %s.%s", dataset_id, table_id)
        except gexc.Conflict:
            log.debug("table %s.%s already exists", dataset_id, table_id)

    # ------------------------------------------------------------------ loads

    def load_frame(
        self,
        frame: pd.DataFrame,
        dataset_id: str,
        table_id: str,
        write_disposition: str = "WRITE_APPEND",
    ) -> int:
        """Load a DataFrame via Parquet, letting the schema widen as needed."""
        if frame.empty:
            return 0
        if self.dry_run:
            log.info(
                "[dry-run] load %d rows x %d cols -> %s.%s",
                len(frame),
                len(frame.columns),
                dataset_id,
                table_id,
            )
            return len(frame)

        buffer = io.BytesIO()
        frame.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)

        config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.PARQUET,
            write_disposition=write_disposition,
            autodetect=True,
            schema_update_options=[
                bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION,
                bigquery.SchemaUpdateOption.ALLOW_FIELD_RELAXATION,
            ],
        )
        target = f"{self.project}.{dataset_id}.{table_id}"
        job = self.client.load_table_from_file(buffer, target, job_config=config)
        job.result()
        if job.errors:
            raise RuntimeError(f"load into {target} failed: {job.errors}")
        return len(frame)

    def load_rows(self, rows: list[dict], dataset_id: str, table_id: str, schema: list) -> int:
        """Load explicit dict rows against a fixed schema (manifest, documents)."""
        if not rows:
            return 0
        if self.dry_run:
            log.info("[dry-run] load %d rows -> %s.%s", len(rows), dataset_id, table_id)
            return len(rows)

        buffer = to_parquet_bytes(pd.DataFrame(rows), schema)
        config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.PARQUET,
            write_disposition="WRITE_APPEND",
            schema=schema,
        )
        target = f"{self.project}.{dataset_id}.{table_id}"
        job = self.client.load_table_from_file(buffer, target, job_config=config)
        job.result()
        if job.errors:
            raise RuntimeError(f"load into {target} failed: {job.errors}")
        return len(rows)

    # ----------------------------------------------------------------- resume

    def already_ingested(self, dataset_id: str) -> tuple[set[str], set[str]]:
        """Return ``(loaded, censused)`` file ids from the manifest.

        Two distinct sets, because they gate different things and conflating them
        corrupts the census:

        * ``loaded`` -- content was actually parsed into a table. Gates whether a
          file needs re-parsing.
        * ``censused`` -- the file has a manifest row of *any* status. Gates
          whether a manifest row should be written at all.

        Using only ``loaded`` for both means statuses that are never 'loaded'
        (duplicate, excluded, metadata_only) get a fresh manifest row on every
        run. Their rows then accumulate, and any view that counts them --
        `drive_insights.duplicates` counting copies per content hash, most
        visibly -- multiplies its numbers by the number of runs.
        """
        if self.dry_run:
            return set(), set()
        query = f"""
            SELECT file_id, ingest_status
            FROM `{self.project}.{dataset_id}.{MANIFEST_TABLE}`
        """
        try:
            rows = list(self.client.query(query).result())
        except gexc.NotFound:
            return set(), set()
        loaded = {r.file_id for r in rows if r.ingest_status == "loaded"}
        censused = {r.file_id for r in rows}
        return loaded, censused

    # -------------------------------------------------------------------- sql

    def sql(self, statement: str, label: str = "") -> object | None:
        """Run one statement. Returns the row iterator, or None in dry-run."""
        if self.dry_run:
            log.info("[dry-run] %s\n%s", label or "sql", statement)
            return None
        log.debug("%s\n%s", label or "sql", statement)
        job = self.client.query(statement)
        return job.result()

    def scalar(self, statement: str, default=None):
        """Run a statement and return the first column of the first row."""
        if self.dry_run:
            log.info("[dry-run] scalar: %s", statement)
            return default
        for row in self.client.query(statement).result():
            return row[0]
        return default

    def list_table_ids(self, dataset_id: str) -> list[str]:
        if self.dry_run:
            return []
        try:
            return [t.table_id for t in self.client.list_tables(f"{self.project}.{dataset_id}")]
        except gexc.NotFound:
            return []
__BQ_PY__'''

MODULES['pipeline.py'] = r'''__PIPELINE_PY__
#!/usr/bin/env python3
"""Load an entire Google Drive into BigQuery.

Three datasets, three jobs:

  drive_raw        file_manifest -- one row per file in Drive, loaded or not.
                   The census. Nothing is silently dropped; anything that fails
                   is recorded here with its error.
  drive_tables     one table per *family* of tabular files. Date-sharded exports
                   (heart_rate_2026-04-05.csv, heart_rate_2026-07-05.csv, ...)
                   collapse into one table with _src_date provenance.
  drive_documents  documents -- extracted text from PDFs, Word, slides, txt/md.

Usage
-----
    python pipeline.py inventory --project PROJECT [--folder FOLDER_ID]
    python pipeline.py plan      --project PROJECT [--folder FOLDER_ID]
    python pipeline.py load      --project PROJECT [--folder FOLDER_ID]

`inventory` writes only the manifest. `plan` prints the table layout without
touching BigQuery. `load` does everything and is safe to re-run: files already
marked loaded in the manifest are skipped.
"""

from __future__ import annotations

import argparse
import datetime as dt
import json
import logging
import os
import sys
import traceback
from pathlib import Path

import pandas as pd
from google.auth import default as google_auth_default
from google.oauth2 import service_account
from googleapiclient.discovery import build

sys.path.insert(0, str(Path(__file__).resolve().parent))

import bq  # noqa: E402
import chunk as chunk_mod  # noqa: E402
import dedupe  # noqa: E402
import drive as drive_mod  # noqa: E402
import graph as gr  # noqa: E402
import preflight  # noqa: E402
import quickinsights as qi  # noqa: E402
import vectorize as vec  # noqa: E402
from classify import build_families, partition  # noqa: E402
from parse import extract_text, read_tabular, reconcile  # noqa: E402

SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/bigquery",
]

RAW_DATASET = "drive_raw"
TABLES_DATASET = "drive_tables"
DOCS_DATASET = "drive_documents"

# Files bigger than this are recorded in the manifest but not parsed, so one
# 4 GB video cannot stall a run over thousands of small files.
MAX_PARSE_BYTES = 512 * 1024 * 1024

# Rows are accumulated per family and flushed in batches to bound memory.
FLUSH_ROWS = 400_000

log = logging.getLogger("drive2bq")


# --------------------------------------------------------------------- helpers


def credentials():
    """Service-account JSON if provided, otherwise Application Default Creds."""
    key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if key_path and Path(key_path).is_file():
        return service_account.Credentials.from_service_account_file(
            key_path, scopes=SCOPES
        )
    creds, _ = google_auth_default(scopes=SCOPES)
    return creds


def now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def _is_stringish(series: pd.Series) -> bool:
    """True for object and for pandas' native string dtypes.

    pandas 2.2 gives ``object``, pandas 3 gives ``str``; checking only for
    ``object`` silently skips every column on newer pandas and leaves the whole
    table as strings.
    """
    return series.dtype == object or pd.api.types.is_string_dtype(series.dtype)


def coerce_types(frame: pd.DataFrame) -> pd.DataFrame:
    """Promote all-string columns to numeric/timestamp where unambiguous.

    Everything is read as string to keep a family's files schema-compatible;
    this puts the real types back once the whole family is merged, so the
    resulting tables are actually queryable with SUM/AVG and date filters.
    """
    for column in frame.columns:
        if column.startswith("_src_") or column == "_ingested_at":
            continue
        series = frame[column]
        if not _is_stringish(series):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue

        numeric = pd.to_numeric(non_null, errors="coerce")
        if numeric.notna().all():
            full = pd.to_numeric(series, errors="coerce")
            # Keep integers narrow when there is no fractional part.
            if (full.dropna() % 1 == 0).all():
                frame[column] = full.astype("Int64")
            else:
                frame[column] = full
            continue

        # Only try dates on values that look like them, so free text that
        # happens to start with a digit is not mangled into 1970.
        sample = non_null.astype(str).head(200)
        if sample.str.match(r"^\d{4}-\d{2}-\d{2}([ T]|$)").mean() > 0.9:
            parsed = pd.to_datetime(series, errors="coerce", format="mixed", utc=True)
            if parsed.notna().sum() >= non_null.shape[0] * 0.99:
                frame[column] = parsed

    # Provenance columns get real types too, so _src_date is filterable as a
    # DATE and _ingested_at as a TIMESTAMP rather than both landing as strings.
    if "_src_date" in frame.columns:
        frame["_src_date"] = pd.to_datetime(
            frame["_src_date"], errors="coerce", format="%Y-%m-%d"
        ).dt.date
    if "_ingested_at" in frame.columns:
        frame["_ingested_at"] = pd.to_datetime(
            frame["_ingested_at"], errors="coerce", utc=True
        )
    return frame


def add_provenance(frame: pd.DataFrame, record: dict, sheet: str) -> pd.DataFrame:
    frame = frame.copy()
    frame.insert(0, "_src_file_id", record["file_id"])
    frame.insert(1, "_src_file_name", record["name"])
    frame.insert(2, "_src_date", record.get("shard_date"))
    frame.insert(3, "_src_sheet", sheet or None)
    frame.insert(4, "_ingested_at", now())
    return frame


def manifest_row(record: dict, **overrides) -> dict:
    row = {
        "file_id": record["file_id"],
        "name": record["name"],
        "path": record["path"],
        "mime_type": record["mime_type"],
        "extension": record["extension"],
        "size_bytes": record["size_bytes"],
        "md5_checksum": record["md5_checksum"],
        "created_time": record["created_time"],
        "modified_time": record["modified_time"],
        "parent_id": record["parent_id"],
        "kind": record["kind"],
        "fmt": record["fmt"],
        "family_stem": record["family_stem"],
        "shard_date": record["shard_date"],
        "content_hash": record.get("content_hash"),
        "duplicate_of": record.get("duplicate_of"),
        "target_dataset": None,
        "target_table": None,
        "ingest_status": "pending",
        "ingest_error": None,
        "row_count": None,
        "ingested_at": now(),
    }
    row.update(overrides)
    return row


def enumerate_drive(args, drive_service=None) -> list[dict]:
    """Enumerate via a mounted path when given, otherwise via the Drive API."""
    if args.local_root:
        log.info("enumerating mounted Drive at %s ...", args.local_root)
        files = list(drive_mod.walk_local(args.local_root))
    else:
        log.info(
            "enumerating Drive%s ...", f" folder {args.folder}" if args.folder else " (all)"
        )
        files = list(drive_mod.walk(drive_service, root_id=args.folder))
    log.info("found %d files", len(files))
    return files


def fetch_bytes(record: dict, args, drive_service) -> tuple[bytes, str]:
    """Read one file's bytes in whichever mode is active."""
    if args.local_root:
        return drive_mod.read_local(record, service=drive_service)
    return drive_mod.download(drive_service, record["file_id"], record["mime_type"])


def needs_api(files: list[dict]) -> bool:
    """True when any file can only be read by exporting through the API."""
    return any(f.get("mime_type") in drive_mod.EXPORT_MIMES for f in files)


def make_drive_service(optional: bool = False):
    """Build a Drive client, tolerating absent credentials in local mode."""
    try:
        return build("drive", "v3", credentials=credentials(), cache_discovery=False)
    except Exception as exc:
        if not optional:
            raise
        log.warning("no Drive credentials (%s); native Google files will be skipped", exc)
        return None


def apply_exclusions(files: list[dict], args) -> tuple[list[dict], list[dict]]:
    """Split off excluded files and report what went."""
    kept, excluded = partition(
        files,
        skip_health=args.skip_health,
        exclude_path=args.exclude_path,
        exclude_family=args.exclude_family,
    )
    if excluded:
        reasons: dict[str, int] = {}
        for record in excluded:
            reasons[record["exclude_reason"]] = reasons.get(record["exclude_reason"], 0) + 1
        for reason, count in sorted(reasons.items(), key=lambda kv: -kv[1]):
            log.info("excluded %d files: %s", count, reason)
    return kept, excluded


def apply_dedup(files: list[dict], args) -> tuple[list[dict], list[dict]]:
    """Drop duplicate content before anything expensive touches it.

    Embedding a document nine times costs nine times as much and, worse, makes
    every cross-link ranking start with a file matching its own copies at
    distance zero. Dedup first or the insight layer produces confident noise.
    """
    if args.no_dedup:
        for record in files:
            record.setdefault("content_hash", record.get("md5_checksum"))
        return files, []
    dedupe.assign_hashes(files)
    canonical, duplicates = dedupe.partition_duplicates(files)
    stats = dedupe.summarize(canonical, duplicates)
    if duplicates:
        log.info(
            "dedup: %d files -> %d distinct (%.2fx), %.1f MiB redundant",
            stats["total"], stats["canonical"], stats["factor"],
            stats["wasted_bytes"] / 1024 / 1024,
        )
        for name, copies, wasted in dedupe.top_duplicated(duplicates, 8):
            log.info("  %2d copies  %7.1f MiB redundant  %s", copies,
                     wasted / 1024 / 1024, name[:60])
    return canonical, duplicates


def summarize(files: list[dict]) -> dict:
    by_kind: dict[str, int] = {}
    bytes_by_kind: dict[str, int] = {}
    for record in files:
        by_kind[record["kind"]] = by_kind.get(record["kind"], 0) + 1
        bytes_by_kind[record["kind"]] = bytes_by_kind.get(record["kind"], 0) + int(
            record["size_bytes"] or 0
        )
    return {"count": len(files), "by_kind": by_kind, "bytes_by_kind": bytes_by_kind}


# ------------------------------------------------------------------- commands


def cmd_inventory(args) -> int:
    drive_service = make_drive_service(optional=bool(args.local_root))
    files = enumerate_drive(args, drive_service)

    stats = summarize(files)
    families = build_families(files)
    print(json.dumps({**stats, "families": len(families)}, indent=2))

    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)
    loader.ensure_dataset(RAW_DATASET)
    loader.ensure_table(RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    rows = [manifest_row(r, ingest_status="inventoried") for r in files]
    loader.load_rows(rows, RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    log.info("manifest written: %d rows", len(rows))

    Path(args.out).write_text(json.dumps(files, indent=2))
    log.info("local inventory cached at %s", args.out)
    return 0


def cmd_plan(args) -> int:
    # Planning from a cached inventory needs no credentials at all, which makes
    # the table layout reviewable before any access is granted.
    if args.from_cache and Path(args.out).is_file():
        files = json.loads(Path(args.out).read_text())
        log.info("loaded %d files from cache %s", len(files), args.out)
    else:
        # A mounted walk needs no credentials at all; the API walk does.
        drive_service = make_drive_service(optional=bool(args.local_root))
        files = enumerate_drive(args, drive_service)
        Path(args.out).write_text(json.dumps(files, indent=2))

    total_found = len(files)
    files, excluded = apply_exclusions(files, args)
    files, duplicates = apply_dedup(files, args)
    stats = summarize(files)
    families = build_families(files)

    if duplicates:
        d = dedupe.summarize(files, duplicates)
        print(f"\nDuplicates: {d['duplicates']} redundant copies of "
              f"{d['canonical']} distinct files ({d['factor']}x), "
              f"{d['wasted_bytes'] / 1024 / 1024:,.1f} MiB")
        for name, copies, wasted in dedupe.top_duplicated(duplicates, 10):
            print(f"  {copies:>3} copies  {wasted / 1024 / 1024:>8.1f} MiB  {name[:56]}")

    print(f"\nFiles: {stats['count']} to load", end="")
    print(f"  ({len(excluded)} excluded of {total_found} found)" if excluded else "")
    for kind, count in sorted(stats["by_kind"].items(), key=lambda kv: -kv[1]):
        mib = stats["bytes_by_kind"].get(kind, 0) / 1024 / 1024
        print(f"  {kind:<10} {count:>6}  ({mib:,.1f} MiB)")

    if excluded:
        excluded_mib = sum(int(f.get("size_bytes") or 0) for f in excluded) / 1024 / 1024
        print(f"\nExcluded ({excluded_mib:,.1f} MiB), still listed in the manifest:")
        stems: dict[str, int] = {}
        for record in excluded:
            stems[record.get("family_stem") or record["kind"]] = (
                stems.get(record.get("family_stem") or record["kind"], 0) + 1
            )
        for stem, count in sorted(stems.items(), key=lambda kv: -kv[1])[:15]:
            print(f"  {stem:<48} {count:>5} files")
        if len(stems) > 15:
            print(f"  ... and {len(stems) - 15} more families")

    print(f"\n{TABLES_DATASET}: {len(families)} tables")
    for name, family in sorted(families.items(), key=lambda kv: -len(kv[1].files)):
        mib = family.total_bytes / 1024 / 1024
        print(f"  {name:<48} {len(family.files):>5} files  ({mib:,.1f} MiB)")

    docs = [f for f in files if f["kind"] == "document"]
    other = [f for f in files if f["kind"] in {"media", "other"}]
    print(f"\n{DOCS_DATASET}.documents: {len(docs)} files (chunked for embedding)")
    print(f"{RAW_DATASET}.file_manifest: {total_found} files "
          f"({len(other)} metadata only, {len(excluded)} excluded)")
    if args.vectorize:
        print(f"{vec.VECTORS_DATASET}.embeddings: document chunks + "
              f"{len(other)} file-metadata rows + rows of chosen tables")
    return 0


def cmd_load(args) -> int:
    drive_service = make_drive_service(optional=bool(args.local_root))
    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)
    preflight.run(loader, args.project,
                  [RAW_DATASET, TABLES_DATASET, DOCS_DATASET], args.location,
                  need_vertex=args.vectorize, connection=vec.CONNECTION_NAME)

    for dataset in (RAW_DATASET, TABLES_DATASET, DOCS_DATASET):
        loader.ensure_dataset(dataset)
    loader.ensure_table(RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)
    loader.ensure_table(DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)
    loader.ensure_table(DOCS_DATASET, bq.CHUNKS_TABLE, bq.CHUNKS_SCHEMA)

    if args.from_cache and Path(args.out).is_file():
        files = json.loads(Path(args.out).read_text())
        log.info("loaded %d files from cache %s", len(files), args.out)
    else:
        files = enumerate_drive(args, drive_service)
        Path(args.out).write_text(json.dumps(files, indent=2))

    total_found = len(files)
    files, excluded = apply_exclusions(files, args)
    files, duplicates = apply_dedup(files, args)

    if args.local_root and drive_service is None and needs_api(files):
        native = sum(1 for f in files if f.get("mime_type") in drive_mod.EXPORT_MIMES)
        log.warning(
            "%d native Google files (Sheets/Docs/Slides) cannot be read from the "
            "mount without credentials; they will be recorded as failed",
            native,
        )

    # --replace truncates each table, so resuming would skip the files whose rows
    # the truncate just destroyed and never put them back. Force a full reload.
    if args.replace and not args.no_resume:
        log.info("--replace implies a full reload; resume disabled")
        args.no_resume = True

    if args.no_resume:
        done, censused = set(), set()
    else:
        done, censused = loader.already_ingested(RAW_DATASET)
    if done or censused:
        log.info(
            "resuming: %d files already loaded, %d already in the manifest",
            len(done), len(censused),
        )

    families = build_families(files)
    manifest: list[dict] = []
    counters = {"loaded": 0, "skipped": 0, "failed": 0, "rows": 0, "chunks": 0}

    # Duplicates are censused with a pointer to their canonical copy, so
    # "where are all the copies of this" stays answerable without re-ingesting.
    for record in duplicates:
        if record["file_id"] in censused:
            continue
        manifest.append(
            manifest_row(
                record,
                ingest_status="duplicate",
                ingest_error=f"duplicate of {record.get('duplicate_of_path')}",
            )
        )

    # Excluded files are still censused, so drive_raw stays a complete picture of
    # the Drive even when the load is deliberately scoped.
    for record in excluded:
        if record["file_id"] in censused:
            continue
        manifest.append(
            manifest_row(
                record,
                ingest_status="excluded",
                ingest_error=record.get("exclude_reason"),
            )
        )

    # ---- tabular families -> drive_tables ---------------------------------
    for table_name, family in sorted(families.items(), key=lambda kv: -len(kv[1].files)):
        pending = [f for f in family.files if f["file_id"] not in done]
        if not pending:
            continue
        log.info(
            "family %s: %d files (%d already loaded)",
            table_name,
            len(pending),
            len(family.files) - len(pending),
        )

        buffered: list[pd.DataFrame] = []
        buffered_rows = 0
        first_write = args.replace

        def flush(frames, replace):
            if not frames:
                return 0
            merged = coerce_types(reconcile(frames))
            disposition = "WRITE_TRUNCATE" if replace else "WRITE_APPEND"
            return loader.load_frame(merged, TABLES_DATASET, table_name, disposition)

        for record in pending:
            size = int(record["size_bytes"] or 0)
            if size > MAX_PARSE_BYTES:
                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="skipped",
                        ingest_error=f"exceeds MAX_PARSE_BYTES ({size} bytes)",
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["skipped"] += 1
                continue
            try:
                data, exported_fmt = fetch_bytes(record, args, drive_service)
                fmt = exported_fmt or record["fmt"]
                sheets = read_tabular(data, fmt, record["name"])
                rows_here = 0
                for sheet_name, frame in sheets:
                    if frame.empty:
                        continue
                    stamped = add_provenance(frame, record, sheet_name)
                    buffered.append(stamped)
                    buffered_rows += len(stamped)
                    rows_here += len(stamped)

                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="loaded",
                        row_count=rows_here,
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["loaded"] += 1
                counters["rows"] += rows_here
            except Exception as exc:  # keep going; record the failure
                log.warning("failed %s (%s): %s", record["name"], record["file_id"], exc)
                manifest.append(
                    manifest_row(
                        record,
                        ingest_status="failed",
                        ingest_error=f"{type(exc).__name__}: {exc}"[:1000],
                        target_dataset=TABLES_DATASET,
                        target_table=table_name,
                    )
                )
                counters["failed"] += 1

            if buffered_rows >= FLUSH_ROWS:
                flush(buffered, first_write)
                first_write = False
                buffered, buffered_rows = [], 0

        flush(buffered, first_write)

    # ---- documents -> drive_documents (+ chunks for embedding) -------------
    doc_rows: list[dict] = []
    chunk_rows: list[dict] = []
    for record in files:
        if record["kind"] != "document" or record["file_id"] in censused:
            continue
        size = int(record["size_bytes"] or 0)
        if size > MAX_PARSE_BYTES:
            manifest.append(
                manifest_row(record, ingest_status="skipped", ingest_error="too large")
            )
            counters["skipped"] += 1
            continue
        try:
            data, exported_fmt = fetch_bytes(record, args, drive_service)
            fmt = exported_fmt or record["fmt"]
            text, pages, method = extract_text(data, fmt)
            doc_rows.append(
                {
                    "file_id": record["file_id"],
                    "name": record["name"],
                    "path": record["path"],
                    "mime_type": record["mime_type"],
                    "fmt": fmt,
                    "size_bytes": record["size_bytes"],
                    "created_time": record["created_time"],
                    "modified_time": record["modified_time"],
                    "page_count": pages,
                    "char_count": len(text),
                    "extraction_method": method,
                    "content": text,
                    "ingested_at": now(),
                }
            )
            # Chunk for embedding. One vector per document would average away
            # the passage you were actually looking for.
            pieces = chunk_mod.chunk_document(record, text)
            for piece in pieces:
                chunk_rows.append({**piece, "ingested_at": now()})
            counters["chunks"] += len(pieces)

            manifest.append(
                manifest_row(
                    record,
                    ingest_status="loaded",
                    row_count=1,
                    target_dataset=DOCS_DATASET,
                    target_table=bq.DOCUMENTS_TABLE,
                )
            )
            counters["loaded"] += 1
        except Exception as exc:
            log.warning("failed doc %s: %s", record["name"], exc)
            manifest.append(
                manifest_row(
                    record,
                    ingest_status="failed",
                    ingest_error=f"{type(exc).__name__}: {exc}"[:1000],
                )
            )
            counters["failed"] += 1

        if len(doc_rows) >= 500:
            loader.load_rows(doc_rows, DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)
            doc_rows = []
        if len(chunk_rows) >= 5000:
            loader.load_rows(chunk_rows, DOCS_DATASET, bq.CHUNKS_TABLE, bq.CHUNKS_SCHEMA)
            chunk_rows = []

    loader.load_rows(doc_rows, DOCS_DATASET, bq.DOCUMENTS_TABLE, bq.DOCUMENTS_SCHEMA)
    loader.load_rows(chunk_rows, DOCS_DATASET, bq.CHUNKS_TABLE, bq.CHUNKS_SCHEMA)

    # ---- everything else: metadata only ------------------------------------
    for record in files:
        if record["kind"] in {"media", "other"} and record["file_id"] not in censused:
            manifest.append(manifest_row(record, ingest_status="metadata_only"))

    loader.load_rows(manifest, RAW_DATASET, bq.MANIFEST_TABLE, bq.MANIFEST_SCHEMA)

    log.info(
        "done: %d loaded, %d skipped, %d failed, %d excluded, "
        "%d rows into %d tables, %d chunks",
        counters["loaded"],
        counters["skipped"],
        counters["failed"],
        len(excluded),
        counters["rows"],
        len(families),
        counters["chunks"],
    )
    counters["excluded"] = len(excluded)
    counters["duplicates"] = len(duplicates)
    counters["found"] = total_found
    print(json.dumps(counters, indent=2))

    if args.vectorize:
        log.info("vectorizing ...")
        rc = run_vectorize(args, loader)
        if rc:
            return rc
    return 1 if counters["failed"] and args.strict else 0


def run_vectorize(args, loader: bq.Loader) -> int:
    """Embed loaded content into drive_vectors.embeddings and index it.

    Idempotent throughout: the queue only ever receives ids that are not already
    embedded, so re-running after adding files embeds just the new material.
    """
    project = args.project
    preflight.run(loader, project, [vec.VECTORS_DATASET], args.location,
                  need_vertex=True, connection=vec.CONNECTION_NAME)

    loader.ensure_dataset(vec.VECTORS_DATASET)
    vec.ensure_connection(project, args.location, dry_run=args.dry_run)

    loader.sql(vec.create_model_sql(project, args.location, args.embedding_model), "model")
    loader.sql(vec.create_embeddings_table_sql(project), "embeddings table")
    loader.sql(vec.create_staging_table_sql(project), "embed queue")

    # Queue everything worth embedding.
    loader.sql(vec.enqueue_from_chunks_sql(project), "queue document chunks")
    loader.sql(vec.enqueue_file_metadata_sql(project), "queue file metadata")

    # Rows are only embedded for tables the caller names; telemetry rows are
    # meaningless as text and would dominate the index.
    for table in args.vectorize_tables or []:
        if table not in loader.list_table_ids(TABLES_DATASET) and not args.dry_run:
            log.warning("no such table %s.%s, skipping", TABLES_DATASET, table)
            continue
        loader.sql(
            vec.enqueue_table_rows_sql(project, table, args.max_rows_per_table),
            f"queue rows of {table}",
        )

    depth = loader.scalar(vec.queue_depth_sql(project), default=0) or 0
    log.info("%s rows queued for embedding", f"{depth:,}")

    # Drain the queue in batches so one oversized query cannot fail the lot.
    batches = 0
    while True:
        if args.dry_run:
            loader.sql(vec.embed_batch_sql(project, args.embed_batch), "embed batch")
            loader.sql(vec.dequeue_embedded_sql(project), "dequeue")
            break
        loader.sql(vec.embed_batch_sql(project, args.embed_batch), "embed batch")
        loader.sql(vec.dequeue_embedded_sql(project), "dequeue")
        batches += 1
        remaining = loader.scalar(vec.queue_depth_sql(project), default=0) or 0
        log.info("batch %d done, %s still queued", batches, f"{remaining:,}")
        if remaining == 0:
            break
        if remaining >= depth and batches > 1:
            # Nothing drained this round: every remaining row is failing.
            log.error(
                "embedding stalled with %s rows queued; inspect "
                "`%s.%s.embed_queue` and the model's status output",
                f"{remaining:,}",
                project,
                vec.VECTORS_DATASET,
            )
            return 1
        depth = remaining
        if batches >= args.max_batches:
            log.warning(
                "stopping after %d batches with %s queued; re-run `vectorize` to continue",
                batches,
                f"{remaining:,}",
            )
            break

    total = loader.scalar(
        f"SELECT COUNT(*) FROM `{project}.{vec.VECTORS_DATASET}.{vec.EMBEDDINGS_TABLE}`",
        default=0,
    ) or 0
    log.info("%s vectors in %s.%s", f"{total:,}", vec.VECTORS_DATASET, vec.EMBEDDINGS_TABLE)

    # A vector index only kicks in above a row threshold; below it BigQuery
    # brute-forces the scan, which is correct but slower.
    if total >= vec.INDEX_MIN_ROWS or args.dry_run:
        loader.sql(vec.create_index_sql(project), "vector index")
    else:
        log.info(
            "skipping vector index: %s rows is below BigQuery's %s-row minimum, "
            "so search will scan instead (same results)",
            f"{total:,}",
            f"{vec.INDEX_MIN_ROWS:,}",
        )

    loader.sql(vec.create_search_function_sql(project), "search function")
    log.info(
        "search with:  SELECT * FROM `%s.%s.search`('your question', 10)",
        project,
        vec.VECTORS_DATASET,
    )
    return 0


def cmd_vectorize(args) -> int:
    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)
    return run_vectorize(args, loader)


def run_crosslink(args, loader: bq.Loader) -> int:
    """Pass 1: search the embeddings against themselves and record the edges."""
    project = args.project
    loader.ensure_dataset(gr.GRAPH_DATASET)
    loader.ensure_dataset(gr.INSIGHTS_DATASET)
    loader.sql(gr.create_state_table_sql(project), "state table")

    log.info("crossing every chunk against every other chunk ...")
    loader.sql(
        gr.build_links_sql(
            project, vec.VECTORS_DATASET, vec.EMBEDDINGS_TABLE,
            neighbours=args.neighbours, max_distance=args.max_link_distance,
        ),
        "build cross links",
    )
    links = loader.scalar(
        f"SELECT COUNT(*) FROM `{project}.{gr.GRAPH_DATASET}.{gr.LINKS_TABLE}`", default=0
    ) or 0
    log.info("%s cross-file links", f"{links:,}")
    loader.sql(gr.record_state_sql(project, "cross_links", links), "state")
    return 0


def run_entities(args, loader: bq.Loader) -> int:
    """Pass 2: extract entities, resolve them, and push them through the edges."""
    project = args.project
    loader.ensure_dataset(gr.GRAPH_DATASET)
    loader.sql(gr.create_mentions_table_sql(project), "mentions table")
    loader.sql(
        gr.create_extractor_model_sql(
            project, vec.VECTORS_DATASET, args.location,
            vec.CONNECTION_NAME, args.text_model,
        ),
        "extractor model",
    )

    pending = loader.scalar(
        gr.pending_extraction_sql(project, vec.VECTORS_DATASET), default=0
    ) or 0
    log.info("%s chunks awaiting entity extraction", f"{pending:,}")

    batches = 0
    while True:
        loader.sql(
            gr.extract_entities_sql(project, vec.VECTORS_DATASET, args.extract_batch),
            "extract entities",
        )
        if args.dry_run:
            break
        batches += 1
        remaining = loader.scalar(
            gr.pending_extraction_sql(project, vec.VECTORS_DATASET), default=0
        ) or 0
        log.info("batch %d done, %s chunks left", batches, f"{remaining:,}")
        if remaining == 0:
            break
        if remaining >= pending:
            # Nothing consumed: every remaining chunk is failing extraction.
            log.error(
                "extraction stalled at %s chunks. The generative SQL in "
                "graph.extract_entities_sql is the thing to check.",
                f"{remaining:,}",
            )
            return 1
        pending = remaining
        if batches >= args.max_batches:
            log.warning("stopping after %d batches, %s left", batches, f"{remaining:,}")
            break

    loader.sql(gr.build_entities_sql(project), "resolve entities")
    mentions = loader.scalar(
        f"SELECT COUNT(*) FROM `{project}.{gr.GRAPH_DATASET}.{gr.MENTIONS_TABLE}`", default=0
    ) or 0
    entities = loader.scalar(
        f"SELECT COUNT(*) FROM `{project}.{gr.GRAPH_DATASET}.{gr.ENTITIES_TABLE}`", default=0
    ) or 0
    log.info("%s mentions resolving to %s entities", f"{mentions:,}", f"{entities:,}")
    loader.sql(gr.record_state_sql(project, "entities", entities), "state")
    return 0


def run_insights(args, loader: bq.Loader) -> int:
    """Build the derived views. Cheap and idempotent -- they are just views."""
    project = args.project
    loader.ensure_dataset(gr.INSIGHTS_DATASET)

    for label, statement in [
        ("entity_timeline", gr.entity_timeline_view_sql(project)),
        ("entity_cooccurrence", gr.cooccurrence_view_sql(project)),
        ("indirect_relations", gr.indirect_relations_view_sql(project)),
        ("file_bridges", gr.file_bridges_view_sql(project)),
        ("entity_gaps", gr.entity_gaps_view_sql(project)),
        ("recent_activity", gr.activity_view_sql(project)),
        ("pipeline_health", gr.health_view_sql(project)),
        ("ask", gr.ask_function_sql(project, vec.VECTORS_DATASET)),
    ]:
        loader.sql(statement, label)
        log.info("built %s.%s", gr.INSIGHTS_DATASET, label)

    if args.insight_feed:
        log.info("writing the narrated insight feed (this one calls Gemini) ...")
        loader.sql(
            gr.insight_feed_sql(project, vec.VECTORS_DATASET, args.feed_size),
            "insight feed",
        )
        rows = loader.scalar(
            f"SELECT COUNT(*) FROM `{project}.{gr.INSIGHTS_DATASET}.insight_feed`", default=0
        ) or 0
        log.info("%s narrated insights", f"{rows:,}")
        loader.sql(gr.record_state_sql(project, "insight_feed", rows), "state")

    log.info("ask a question:  SELECT * FROM `%s.%s.ask`('...', 12)",
             project, gr.INSIGHTS_DATASET)
    return 0


def run_quickinsights(args, loader: bq.Loader) -> int:
    """Insights needing no embeddings, no Vertex, and no LLM."""
    project = args.project
    preflight.run(loader, project, [qi.INSIGHTS_DATASET, RAW_DATASET], args.location)
    loader.ensure_dataset(qi.INSIGHTS_DATASET)
    for label, statement in qi.all_views(project):
        loader.sql(statement, label)
        log.info("built %s.%s", qi.INSIGHTS_DATASET, label)

    if args.dry_run:
        return 0

    rows = list(loader.client.query(qi.headline_sql(project)).result())
    if rows:
        r = rows[0]
        print("\n" + "=" * 62)
        print("  DRIVE AT A GLANCE")
        print("=" * 62)
        print(f"  files                {r.files:>12,}")
        print(f"  distinct contents    {r.distinct_contents:>12,}")
        print(f"  redundant copies     {r.duplicate_files:>12,}"
              f"   ({r.duplicate_gib} GiB)")
        print(f"  total size           {r.total_gib:>12} GiB")
        print(f"  documents            {r.documents:>12,}")
        print(f"  tabular              {r.tabular:>12,}")
        print(f"  media                {r.media:>12,}")
        print(f"  excluded             {r.excluded:>12,}")
        print(f"  failed               {r.failed:>12,}")
        print(f"  date range           {r.earliest} .. {r.latest}")
        print("=" * 62)

    print("\n  worst duplication:")
    for row in loader.client.query(f"""
        SELECT name, copies, ROUND(bytes_wasted / 1048576, 1) AS mib
        FROM `{project}.{qi.INSIGHTS_DATASET}.duplicates`
        ORDER BY bytes_wasted DESC LIMIT 12
    """).result():
        print(f"    {row.copies:>3} copies  {row.mib:>9,.1f} MiB  {row.name[:52]}")

    print("\n  documents most connected by shared rare terms:")
    for row in loader.client.query(f"""
        SELECT a_name, b_name, shared_terms, score, crosses_folder
        FROM `{project}.{qi.INSIGHTS_DATASET}.term_bridges`
        ORDER BY score DESC LIMIT 12
    """).result():
        flag = " *" if row.crosses_folder else "  "
        print(f"   {flag} {row.score:>7.2f}  {row.shared_terms:>4} terms  "
              f"{row.a_name[:28]:<30} <-> {row.b_name[:28]}")
    print("\n  (* = the two files live in different top-level folders)")
    return 0


def cmd_quickinsights(args) -> int:
    return run_quickinsights(args, bq.Loader(args.project, args.location,
                                            dry_run=args.dry_run))


def cmd_crosslink(args) -> int:
    return run_crosslink(args, bq.Loader(args.project, args.location, dry_run=args.dry_run))


def cmd_entities(args) -> int:
    return run_entities(args, bq.Loader(args.project, args.location, dry_run=args.dry_run))


def cmd_insights(args) -> int:
    return run_insights(args, bq.Loader(args.project, args.location, dry_run=args.dry_run))


def cmd_activate(args) -> int:
    """Everything downstream of the vector table, in order, then the schedules.

    This is the command to put on a timer: crossing, entities, insights.
    """
    loader = bq.Loader(args.project, args.location, dry_run=args.dry_run)
    for step in (run_crosslink, run_entities, run_insights):
        rc = step(args, loader)
        if rc:
            return rc

    print("\nTo let BigQuery refresh this on its own, run these once:\n")
    for name, command in gr.schedule_commands(args.project, args.location):
        print(f"# {name}\n{command}\n")
    print(
        "The ingest step still needs somewhere with Drive access (re-run the\n"
        "notebook, or put `load` on Cloud Run + Cloud Scheduler). Everything\n"
        "above is pure SQL, so BigQuery drives it with no machine of yours."
    )
    return 0


# ----------------------------------------------------------------------- main


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "command",
        choices=["inventory", "plan", "load", "quickinsights", "vectorize",
                 "crosslink", "entities", "insights", "activate"],
    )
    parser.add_argument("--project", required=True, help="GCP project id")
    parser.add_argument("--folder", help="Drive folder id to limit the walk to")
    parser.add_argument(
        "--local-root",
        help="path to an already-mounted Drive (e.g. /content/drive/MyDrive) to "
        "read from the filesystem instead of the Drive API",
    )
    parser.add_argument("--location", default="US", help="BigQuery dataset location")
    parser.add_argument("--out", default="drive_inventory.json", help="inventory cache")
    parser.add_argument("--from-cache", action="store_true", help="reuse cached inventory")
    parser.add_argument("--dry-run", action="store_true", help="no BigQuery writes")
    parser.add_argument("--replace", action="store_true", help="truncate tables first")
    parser.add_argument("--no-resume", action="store_true", help="reload everything")
    parser.add_argument("--strict", action="store_true", help="exit 1 on any failure")
    parser.add_argument("-v", "--verbose", action="store_true")

    scope = parser.add_argument_group("scope")
    scope.add_argument(
        "--skip-health",
        action="store_true",
        help="exclude wearable/health telemetry (Fitbit-style per-day exports). "
        "Excluded files are still recorded in the manifest as 'excluded'.",
    )
    scope.add_argument(
        "--no-dedup",
        action="store_true",
        help="load every copy of duplicated content (default is to keep one and "
        "record the rest in the manifest as 'duplicate')",
    )
    scope.add_argument("--exclude-path", help="regex; exclude files whose path matches")
    scope.add_argument("--exclude-family", help="regex; exclude matching table families")

    vector = parser.add_argument_group("vectors")
    vector.add_argument(
        "--vectorize",
        action="store_true",
        help="after loading, embed content into drive_vectors.embeddings",
    )
    vector.add_argument(
        "--vectorize-tables",
        nargs="*",
        metavar="TABLE",
        help="also embed the rows of these drive_tables tables (descriptive "
        "tables only -- telemetry rows are meaningless as text)",
    )
    vector.add_argument(
        "--embedding-model",
        default=vec.DEFAULT_ENDPOINT,
        help=f"Vertex embedding endpoint (default {vec.DEFAULT_ENDPOINT}; "
        "gemini-embedding-001 is newer but 3072-dim)",
    )
    vector.add_argument(
        "--embed-batch", type=int, default=vec.EMBED_BATCH_ROWS, help="rows per embed query"
    )
    vector.add_argument(
        "--max-batches", type=int, default=500, help="stop after this many embed batches"
    )
    vector.add_argument(
        "--max-rows-per-table",
        type=int,
        default=50_000,
        help="cap on rows embedded per table",
    )

    cross = parser.add_argument_group("crossing and insights")
    cross.add_argument(
        "--neighbours",
        type=int,
        default=gr.NEIGHBOURS_PER_CHUNK,
        help=f"cross-file neighbours kept per chunk (default {gr.NEIGHBOURS_PER_CHUNK})",
    )
    cross.add_argument(
        "--max-link-distance",
        type=float,
        default=gr.MAX_LINK_DISTANCE,
        help=f"cosine distance above which a link is discarded "
        f"(default {gr.MAX_LINK_DISTANCE})",
    )
    cross.add_argument(
        "--text-model",
        default=gr.DEFAULT_TEXT_ENDPOINT,
        help=f"Gemini endpoint for extraction and narration "
        f"(default {gr.DEFAULT_TEXT_ENDPOINT})",
    )
    cross.add_argument(
        "--extract-batch", type=int, default=2000, help="chunks per extraction query"
    )
    cross.add_argument(
        "--insight-feed",
        action="store_true",
        help="also have Gemini narrate the strongest bridges (costs tokens)",
    )
    cross.add_argument(
        "--feed-size", type=int, default=40, help="bridges to narrate"
    )

    args = parser.parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s %(levelname)-7s %(message)s",
    )

    handler = {
        "inventory": cmd_inventory,
        "plan": cmd_plan,
        "load": cmd_load,
        "quickinsights": cmd_quickinsights,
        "vectorize": cmd_vectorize,
        "crosslink": cmd_crosslink,
        "entities": cmd_entities,
        "insights": cmd_insights,
        "activate": cmd_activate,
    }[args.command]
    try:
        return handler(args)
    except preflight.PreflightError as exc:
        # A configuration problem with a known fix; a traceback would only bury it.
        print(f"\nCannot proceed:\n\n  {exc}\n", file=sys.stderr)
        return 3
    except Exception:
        traceback.print_exc()
        return 2


if __name__ == "__main__":
    raise SystemExit(main())
__PIPELINE_PY__'''

for _name, _text in MODULES.items():
    # Strip the heredoc markers off both ends.
    _body = _text.split('\n', 1)[1].rsplit('\n', 1)[0]
    pathlib.Path(_name).write_text(_body)

print('wrote:', ', '.join(MODULES))

## 3. Mount Drive and authenticate

Two consent prompts: one to mount Drive, one for Google Cloud. Both are your own account — nothing is shared with anyone.

In [ ]:
from google.colab import auth, drive

drive.mount('/content/drive')
auth.authenticate_user()
print('Drive mounted and Google Cloud authenticated')

## 4. Preview the plan

Walks the mount and prints exactly what it would build. **No BigQuery writes.**

`--skip-health` excludes wearable telemetry — the per-day Fitbit-style exports (`heart_rate_*`, `body_temperature_*`, `micro_motion_*`, steps, calories, SpO2 and so on). These are millions of rows of near-zero information per row and nothing worth embedding.

**What `--skip-health` does *not* exclude:** your clinical notes, medical PDFs, and anything under `Medical Notes`. Those are documents, not telemetry, and they are kept and vectored. If you want those out too, add `--exclude-path 'Medical Notes'`.

Excluded files are still recorded in `drive_raw.file_manifest` as `excluded`, so the census stays a complete picture of the Drive.

In [ ]:
PROJECT = 'pelagic-gist-505800-b9'
ROOT = '/content/drive/MyDrive'
LOCATION = 'US'

!python pipeline.py plan --project $PROJECT --local-root "$ROOT" \
    --skip-health --vectorize

## 5. Load

Review the table list above first. This is the step that writes to BigQuery.

Safe to re-run: file ids already marked loaded in the manifest are skipped, so an interrupted run resumes instead of duplicating rows. A file that fails is recorded in the manifest and does not abort the run.

Add `--dry-run` to log every operation without executing it. Add `--replace` to truncate the tables first rather than appending. Drop `--skip-health` to bring the telemetry in as well.

In [ ]:
!python pipeline.py load --project $PROJECT --local-root "$ROOT" \
    --location $LOCATION --skip-health --from-cache

## 6. Insights — free, instant, no setup

**Run this before the Vertex steps.** Everything here is plain SQL over the manifest and the extracted text: no embeddings, no model, no connection to configure. Seconds to run, cents to bill, and nothing that can be misconfigured. If you only ever run one insight cell, run this one.

It builds eight views in `drive_insights`:

| View | Answers |
|---|---|
| `duplicates` | Which files exist as many copies, and how much space that wastes. |
| `census` | What is actually in here, by kind and format, with date ranges. |
| `storage` | Where the bytes are, per folder, and what fraction is redundant. |
| `name_clusters` | Near-duplicates content hashing misses — `report.pdf` vs `report (1).pdf`, version chains. |
| `file_timeline` | Corpus activity by month and folder. |
| `doc_terms` | Rare-term index over document text. |
| `term_bridges` | **Documents crossed against each other by shared rare terms — no vectors needed.** |
| `distinctive_terms` | What each document is about, by its rarest frequent terms. |

`term_bridges` is the interesting one. It crosses documents using shared *rare* terms, scored by inverse document frequency, so a pair sharing one very rare term (a case number, an unusual surname) outranks a pair sharing several common ones. Terms appearing in more than 20% of documents are dropped as boilerplate. Less subtle than embeddings — it will not catch a paraphrase — but for records work, shared *identifiers* are usually what matter, and this costs nothing.

In [ ]:
!python pipeline.py quickinsights --project $PROJECT --location $LOCATION

### The duplication problem, in detail

Worth looking at closely before you spend anything on embedding. Sampling this Drive's spreadsheets found **31 distinct documents existing as 100 files** — `PGY-3.xlsx` nine times over — and a 240 MB textbook PDF stored twice.

`load` deduplicates by content hash before parsing, so you are not billed to embed the same document nine times. More importantly, without dedup every cross-file link would rank identical copies (distance ≈ 0) above every genuine connection, and the expensive layer would produce confident noise.

Every copy is still recorded in the manifest as `duplicate` with a pointer to the canonical one, so nothing is lost and \"where are all the copies\" stays answerable.

In [ ]:
client = bigquery.Client(project=PROJECT) if 'client' not in dir() else client

client.query(f'''
    SELECT name, copies,
           ROUND(bytes_each / 1048576, 2)   AS mib_each,
           ROUND(bytes_wasted / 1048576, 1) AS mib_wasted,
           ARRAY_LENGTH(top_folders)        AS folders_spanned,
           paths
    FROM `{PROJECT}.drive_insights.duplicates`
    ORDER BY bytes_wasted DESC
    LIMIT 25
''').to_dataframe()

### Crossed without a single embedding

Document pairs joined by shared rare terms. `*` in `crosses_folder` means the two files sit in different top-level folders — a connection that cuts across how you organised things, which is usually where the surprise is.

In [ ]:
client.query(f'''
    SELECT a_name, b_name, shared_terms, score, crosses_folder,
           rarest_shared
    FROM `{PROJECT}.drive_insights.term_bridges`
    ORDER BY score DESC
    LIMIT 25
''').to_dataframe()

### Where your storage actually goes

In [ ]:
client.query(f'''
    SELECT top_folder, kind, files,
           ROUND(bytes / 1048576, 1)           AS mib,
           duplicate_files,
           ROUND(redundant_fraction * 100, 1)  AS pct_redundant
    FROM `{PROJECT}.drive_insights.storage`
    WHERE bytes > 0
    ORDER BY bytes DESC
    LIMIT 30
''').to_dataframe()

---

**Everything from here on is optional.** Steps 7–13 add semantic search and the entity graph, which are more powerful but need a Vertex connection and cost real money per token. Stop here if the views above answer your questions.

## 7. Connect BigQuery to Vertex AI

Embeddings are generated **inside** BigQuery by `ML.GENERATE_EMBEDDING`, so the text never leaves BigQuery and there is no client-side embedding loop.

That needs a one-time CLOUD_RESOURCE connection, and the connection's own service account needs Vertex access. This cell creates the connection, reads back its service account, and grants it `roles/aiplatform.user`.

It also enables the Vertex AI API if it is not already on. Both steps need you to be an owner/editor on the project.

In [ ]:
import json, subprocess

CONNECTION = 'drive_vertex'

!gcloud services enable aiplatform.googleapis.com --project $PROJECT --quiet

# Create the connection (harmless if it already exists).
!bq mk --connection --location=$LOCATION --project_id=$PROJECT \
    --connection_type=CLOUD_RESOURCE $CONNECTION 2>/dev/null || true

info = subprocess.run(
    ['bq', 'show', '--format=json', '--connection',
     f'{PROJECT}.{LOCATION}.{CONNECTION}'],
    capture_output=True, text=True)
assert info.returncode == 0, info.stderr
SA = json.loads(info.stdout)['cloudResource']['serviceAccountId']
print('connection service account:', SA)

# Let that service account call Vertex.
!gcloud projects add-iam-policy-binding $PROJECT \
    --member=serviceAccount:$SA --role=roles/aiplatform.user --quiet
print('granted roles/aiplatform.user')

## 8. Vectorize

Embeds three things into one searchable table, `drive_vectors.embeddings`:

| Source | What gets embedded |
|---|---|
| `document_chunk` | Every PDF/Word/slide/text document, split into ~2000-char overlapping passages so a hit points at a findable location. |
| `file_metadata` | Name, folder, type and date of every image, video and binary. A photo has no extractable text, but *"that scan from the hospital"* still needs to be findable. |
| `table_row` | Rows of the tables you name in `--vectorize-tables` — for spreadsheets of names, dates and notes. |

One index over all of it means one query searches the whole Drive.

The IAM grant above can take a minute to propagate. If this fails with a permission error, wait and re-run — it is idempotent and resumes from the queue.

In [ ]:
# Name the descriptive tables whose individual rows are worth searching.
# Check the step-4 output for the real table names and edit this list.
VECTOR_TABLES = 'pgy_1 pgy_2 pgy_3 ops_decertification_list_4_19_23'

!python pipeline.py vectorize --project $PROJECT --location $LOCATION \
    --vectorize-tables $VECTOR_TABLES

## 9. Check the result

Row counts per table, then anything that did not load and why.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

print('--- rows per table ---')
tables = list(client.list_tables(f'{PROJECT}.drive_tables'))
for table in sorted(tables, key=lambda t: t.table_id):
    meta = client.get_table(table.reference)
    print(f'{table.table_id:<44} {meta.num_rows:>12,} rows'
          f'  {len(meta.schema):>3} cols')

print()
print('--- ingest status ---')
for row in client.query(f'''
    SELECT ingest_status, COUNT(*) AS files, SUM(row_count) AS rows
    FROM `{PROJECT}.drive_raw.file_manifest`
    GROUP BY ingest_status ORDER BY files DESC
'''):
    print(f'{row.ingest_status:<16} {row.files:>6} files  {row.rows or 0:>12,} rows')

### Anything that failed

Empty result means everything loaded.

In [ ]:
for row in client.query(f'''
    SELECT name, kind, fmt, size_bytes, ingest_error
    FROM `{PROJECT}.drive_raw.file_manifest`
    WHERE ingest_status = 'failed'
    ORDER BY size_bytes DESC LIMIT 50
'''):
    print(f'{row.name[:52]:<54} {row.ingest_error}')

### What got vectored

In [ ]:
for row in client.query(f'''
    SELECT source_kind, COUNT(*) AS vectors,
           COUNT(DISTINCT file_id) AS files,
           ANY_VALUE(ARRAY_LENGTH(embedding)) AS dims
    FROM `{PROJECT}.drive_vectors.embeddings`
    GROUP BY source_kind ORDER BY vectors DESC
'''):
    print(f'{row.source_kind:<18} {row.vectors:>8,} vectors  '
          f'{row.files:>6,} files  {row.dims} dims')

# Anything still queued failed to embed and can be retried.
left = list(client.query(f'''
    SELECT COUNT(*) AS n FROM `{PROJECT}.drive_vectors.embed_queue`
'''))[0].n
print(f'\nstill queued: {left:,}' if left else '\nqueue empty — all embedded')

## 10. Semantic search

The payoff. Ask in plain language; it searches documents, spreadsheet rows and filenames at once. `distance` is cosine — lower is closer.

In [ ]:
QUESTION = 'academic probation appeal deadline'

client.query(f'''
    SELECT name, source_kind, ROUND(distance, 4) AS distance,
           SUBSTR(content, 1, 300) AS excerpt
    FROM `{PROJECT}.drive_vectors.search`(@q, 10)
    ORDER BY distance
''', job_config=bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter('q', 'STRING', QUESTION)]
)).to_dataframe()

Try others — `'residency evaluation concerns'`, `'decertification'`, `'what did the clinic note say about sleep'`. Because filenames are vectored too, searches like `'hospital scan photo'` surface images that contain no text at all.

## 11. Cross the data against itself

Search only answers what you thought to ask. This step makes the corpus surface connections nobody queried for, in two passes:

**Pass 1 — crossing.** Every chunk is searched against every other chunk, and nearest neighbours *in different files* are recorded as edges. A passage in a probation letter and a row in a residency evaluation that discuss the same thing become an explicit link, with no query typed by anyone.

**Pass 2 — crossing the crossings.** Entities are extracted from each chunk, then pushed *through* those edges. Two people who never appear in the same file, but who are each central to files that link to one another, get surfaced as related. That relationship exists in no single document — only in the geometry between them.

Both passes are resumable and skip work already done, so re-running after adding files only processes the new material.

In [ ]:
# Pass 1: edges between chunks in different files.
!python pipeline.py crosslink --project $PROJECT --location $LOCATION

In [ ]:
# Pass 2: entities, then the derived views. --insight-feed additionally has
# Gemini write up the strongest bridges in plain language (costs tokens).
!python pipeline.py entities --project $PROJECT --location $LOCATION
!python pipeline.py insights --project $PROJECT --location $LOCATION \
    --insight-feed

## 12. The entity graph

Seven views in `drive_insights`. Each cell below is one question you could not ask of the raw Drive.

### Where two parts of your Drive meet

File pairs joined by many semantic links. `crosses_folder` marks pairs from *different* top-level folders — a link that crosses how you organised things is usually the interesting one.

In [ ]:
client.query(f'''
    SELECT a_name, b_name, link_count,
           ROUND(closest_distance, 4) AS closest,
           crosses_folder
    FROM `{PROJECT}.drive_insights.file_bridges`
    ORDER BY crosses_folder DESC, link_count DESC
    LIMIT 25
''').to_dataframe()

### Relationships that exist in no single file

Entity pairs that never share a passage, but sit at either end of links between different files. This is the crossing-the-crossings output.

In [ ]:
client.query(f'''
    SELECT entity_a, entity_b, type_a, type_b, bridge_count,
           ROUND(closest_distance, 4) AS closest
    FROM `{PROJECT}.drive_insights.indirect_relations`
    ORDER BY bridge_count DESC, closest_distance
    LIMIT 30
''').to_dataframe()

### Who and what recurs

Entities ranked by how many *distinct files* mention them. Spanning many files matters more than being repeated inside one.

In [ ]:
client.query(f'''
    SELECT display_name, entity_type, file_count, mention_count
    FROM `{PROJECT}.drive_graph.entities`
    WHERE entity_type IN ('PERSON', 'ORG', 'CASE_NUMBER')
    ORDER BY file_count DESC, mention_count DESC
    LIMIT 30
''').to_dataframe()

### A chronology assembled across sources

Pick an entity from the table above and get every dated appearance of it, wherever it came from. `date_source` says where each date was found: `in_text` (a date in the same passage), `filename`, or `file_mtime`.

In [ ]:
ENTITY = 'probation'   # substring, casefolded

client.query(f'''
    SELECT event_date, date_source, entity_text, entity_type,
           file_name, source_kind
    FROM `{PROJECT}.drive_insights.entity_timeline`
    WHERE entity_norm LIKE CONCAT('%', LOWER(@e), '%')
    ORDER BY event_date
    LIMIT 100
''', job_config=bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter('e', 'STRING', ENTITY)]
)).to_dataframe()

### Narrated bridges

Gemini's read on the strongest cross-file matches. It was told to be blunt and to label most matches `COINCIDENTAL`, so `looks_meaningful = true` is a real signal rather than flattery. Empty if you skipped `--insight-feed`.

In [ ]:
client.query(f'''
    SELECT a_name, b_name, looks_meaningful,
           ROUND(distance, 4) AS distance, assessment
    FROM `{PROJECT}.drive_insights.insight_feed`
    ORDER BY looks_meaningful DESC, distance
    LIMIT 25
''').to_dataframe()

### Coverage gaps

Entities present in your documents but absent from every spreadsheet, or the reverse. Either a data gap or a finding — worth seeing either way.

In [ ]:
client.query(f'''
    SELECT display_name, entity_type, coverage,
           in_documents, in_tables, file_count
    FROM `{PROJECT}.drive_insights.entity_gaps`
    WHERE coverage != 'both' AND file_count >= 2
    ORDER BY file_count DESC
    LIMIT 30
''').to_dataframe()

### Ask it anything

Retrieval-augmented answering over the whole Drive: semantic search fetches the passages, Gemini answers from them and cites the filenames. It is instructed to say when the corpus does not contain the answer rather than inventing one — check the `sources` column against the claim regardless.

In [ ]:
for row in client.query(f'''
    SELECT answer, sources
    FROM `{PROJECT}.drive_insights.ask`(@q, 12)
''', job_config=bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter(
        'q', 'STRING',
        'What concerns were raised about the residency, and by whom?')]
)):
    print(row.answer)
    print('\nsources:', ', '.join(row.sources or []))

## 13. Keep it alive

Everything downstream of the vector table is pure SQL, so BigQuery can refresh it on a timer with no machine of yours involved. `activate` runs all three passes and then prints the `bq query --schedule` commands to register them.

The **ingest** step is the one exception — it needs somewhere with Drive access. Either re-run this notebook when you have added files, or put `load` on Cloud Run behind Cloud Scheduler for true hands-off operation.

In [ ]:
!python pipeline.py activate --project $PROJECT --location $LOCATION \
    --insight-feed

### Is it current?

Each stage records a watermark when it runs. `stale` flags anything that has not run in 48 hours — the difference between a live system and one that quietly stopped.

In [ ]:
client.query(f'''
    SELECT stage, last_run, rows_seen, hours_since, stale
    FROM `{PROJECT}.drive_insights.pipeline_health`
    ORDER BY last_run DESC
''').to_dataframe()

---

**Note on native Google files.** Sheets, Docs and Slides are not real files on the mount — Drive represents them as small stub files with no data. The pipeline recovers their file id from the stub and exports them through the Drive API. If Colab's credentials lack the Drive scope they are recorded as `failed` in the manifest with a clear reason, and everything else still loads. Uploaded `.xlsx` files are real files and are unaffected.